# taskb_adapter_dev

Verified storagefix source of the reused adapter seed42 checkpoint. See [Task B guide](README_taskb.md) for report evidence, execution order and historical-source limitations.


In [ ]:
# Hyperparameter giữ nguyên; run mới giữ log cũ, full cache cũ chỉ đọc nếu fingerprint khớp.
CFG = {'preparation_root': '/content/drive/MyDrive/ĐATN/data/processed/benchmark_preparation/FashionIQ/fashioniq_prep_v1_20261103', 'development_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_development/FashionIQ/tme_dev_v1_seed42', 'output_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ', 'experiment_id': 'retrieval_only_dev_v1_base42_adapter42_checked_v2_storagefix', 'max_epochs': 10, 'batch_size': 32, 'encode_batch_size': 16, 'eval_batch_size': 32, 'cache_shard_size': 128, 'adapter_width': 128, 'adapter_heads': 4, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'retrieval_temperature': 0.07, 'gradient_clip': 1.0, 'checkpoint_every_steps': 64, 'resume': True, 'amp': False, 'tf32': False, 'require_cuda': True, 'baseline_max_metric_deviation_pp': 0.01, 'gate_equivalence_atol': 1e-06, 'local_root': '/content/visfit_tme_adapter_development', 'vision_dim': 768, 'base_seed': 42, 'adapter_seed': 42, 'train_count': 16204, 'trial_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_trial/FashionIQ/adapter_trial_v1_seed42', 'patience': 3, 'export_top_k': 50, 'verified_full_cache_source': {'run_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v1', 'local_root': '/content/visfit_tme_adapter_development/retrieval_only_dev_v1_base42_adapter42_checked_v1'}}
RUN_FULL_TRAINING = False


## 1. Bootstrap và Drive
Dependencies trước import model; guard version trên disk lẫn module đã nạp, không tự nâng torch/CUDA. Nếu yêu cầu restart: lưu notebook/traceback trước, restart một lần rồi chạy lại từ đầu. Drive mount cần bạn cấp quyền khi được hỏi.

In [ ]:
# Chỉ stdlib trước import model; bootstrap fix: dependency-owner-and-loaded-version-v2.
import sys, subprocess, importlib.util, importlib.metadata, json
from pathlib import Path
BOOTSTRAP = {'python':sys.version,'installs':[],'pins':{},'passed':False,'bootstrap_fix':'dependency-owner-and-loaded-version-v2'}
local=Path('/content/visfit_tme_adapter_development_bootstrap');local.mkdir(parents=True,exist_ok=True)
BOOTSTRAP['loaded_before_bootstrap']={k:{'version':getattr(v,'__version__',None),'path':getattr(v,'__file__',None)} for k,v in list(sys.modules.items()) if k in ('torch','transformers','tokenizers','huggingface_hub')}
PINS = {'transformers':'4.57.1','tokenizers':'0.22.1','huggingface-hub':'0.36.0'}
for package in ('torch','numpy','pandas','Pillow'):
    try: BOOTSTRAP['pins'][package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError('Colab base package missing: '+package+'; use a standard GPU runtime') from None
needed=[]
for package,version in PINS.items():
    try: installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: installed=None
    if installed!=version:needed.append(package+'=='+version)
    module={'huggingface-hub':'huggingface_hub'}.get(package,package)
    loaded=any(n==module or n.startswith(module+'.') for n in sys.modules)
    loaded_version=getattr(sys.modules.get(module),'__version__',None)
    if loaded and (installed!=version or loaded_version!=version):
        BOOTSTRAP.update(restart_required=True,blocked_package=package,installed_version=installed,loaded_version=loaded_version)
        (local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
        raise RuntimeError('A dependency requiring a version change is already loaded: '+package+'. Save the notebook and use Runtime > Restart session, then run from the top. No automatic install/restart loop.')
    BOOTSTRAP['pins'][package]=version
if needed:
    p=subprocess.run([sys.executable,'-m','pip','install','--quiet',*needed],capture_output=True,text=True)
    if p.returncode:raise RuntimeError('Dependency installation failed; stopped before importing models. No torch/CUDA upgrade requested.')
    BOOTSTRAP['installs']=needed
check=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
BOOTSTRAP['pip_check']={'returncode':check.returncode,'report':check.stdout+check.stderr}
# Scope by dependent package owner, not a substring of the dependency mentioned in the error.
# Gradio/Diffusers are not used by this pilot; their conflicts remain visible in the report.
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
pending=['torch','torchvision','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow','safetensors','packaging']
core=set()
while pending:
    owner=canonicalize_name(pending.pop())
    if owner in core:continue
    core.add(owner)
    for raw in importlib.metadata.requires(owner) or []:
        req=Requirement(raw)
        if req.marker is None or req.marker.evaluate({'extra':''}):pending.append(req.name)
lines=[line for line in (check.stdout+check.stderr).splitlines() if line.strip() and 'No broken requirements found' not in line]
critical=[line for line in lines if canonicalize_name(line.split()[0]) in core]
BOOTSTRAP['dependency_scope']={'core_packages':sorted(core),'critical_conflicts':critical,'outside_pilot_conflicts':[line for line in lines if line not in critical]}
local=Path('/content/visfit_tme_adapter_development_bootstrap');local.mkdir(parents=True,exist_ok=True)
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
if check.returncode and critical:raise RuntimeError('pip check found TME dependency conflicts: '+'; '.join(critical))
probe=subprocess.run([sys.executable,'-c',
    'import torch,transformers; from transformers import CLIPModel,CLIPTokenizer,CLIPImageProcessor; from transformers.generation import GenerationMixin; print(transformers.__version__); print(transformers.__file__)'],capture_output=True,text=True)
BOOTSTRAP['import_probe']={'returncode':probe.returncode,'stdout':probe.stdout,'traceback':probe.stderr}
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
if probe.returncode:raise RuntimeError('CLIP dependency import failed in a clean subprocess. Full traceback: /content/visfit_tme_adapter_development_bootstrap/bootstrap_report.json. Diagnose before dataset/training.')
BOOTSTRAP['passed']=True
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
print('Bootstrap đạt; giữ nguyên torch/CUDA:',BOOTSTRAP['pins']['torch'])
print('TME dependency conflicts:',len(critical),'| clean CLIP import:',probe.returncode==0,'| packages installed:',BOOTSTRAP['installs'])
if check.returncode:print('pip check có xung đột ngoài dependency graph của pilot; giữ nguyên đầy đủ trong bootstrap_report.json. Không coi toàn môi trường là pip-clean.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PREPARATION_ANCHORS = {'data_audit.json': 'fbaa7eae7b99c5e5aeeddcb0bc6027d3870e8c4f9be5658559caa95662321e95', 'feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342', 'extraction_contract.json': 'bbd53b94172aa5c07a358a0697163c620a823b297c6fec6209167bdb59d1f0a1', 'training_plan.json': 'e5496feae33ab0e9dc832e29a1a9097b2d7fed6c2683bd9d4314d6e4973bdc21', 'development_gallery.csv': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'development_val.csv': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5', 'protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb', 'development_train.csv': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'feature_views/development_gallery.csv': '283199425c5fc88fb8e063ea28be629e611e44513689e97f22ee3d0ec5a33927', 'feature_views/development_val.csv': 'cbd8d7b72fa7fb4e3c38d64f3bb8ceeeba73be946e9918c5a77c856a42217619', 'feature_views/development_train.csv': '6f7a5e6ab8792dd8fd08d6fea8c472a9d9b56b59af38ba146467d797f8a98ff0'}
SOURCE_LOCKS = {'config.json': 'b3d48c8550a8f36ee1bbbc575878d10ac10e416509b8a599ba23019c7561935f', 'best.json': '9ec6117b48e0df64e49247d9f41da1cc035c1a504e00551abc1aad654aadbca1', 'architecture_contract.json': '49c7db63abcb71ce0d7be083aa3a7b82c89e290adabbd77264226e9ee31b627b', 'pilot_config.json': '3b6097620b76a2789579375987e64594fc82637e7e2a3ab271f9ab4145243685', 'runtime_contract.json': '02696d2a36995ffd90791671203673b689bede949c5f05d92c20c4741e65ee2d', 'cache_binding.json': '78f3e73eed98fa2c6b27edd61fe16c542f925a9157cced035e8dd6aef858093d', 'pilot_manifest_hashes.json': '6d8198b224e4e07242bbc39dcef7e189e06b9a1d50a416f6e5013572debe657c', 'metric_history.json': '253717bafb9c38d8dd1f0cf484d78b6165c2708233afec225bc7a905e3d8829e', 'epoch_03_per_query.csv': '00712dcc75ac63ded64e75465e566c1c183365eda3df7b7f832ddc0f411568a2'}
EXPECTED_BASE_MARKER = {'phase': 'conventional', 'sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'signature': 'c6cf0642cdcf5b7c7c5fc8fe6246fd7f0c668377036fefb12d45cfb1c10dc3d3', 'step': 12153}
EXPECTED_METRICS = {'Mean': 72.6305734380787, 'gallery_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'n_query': 1796, 'overall': {'R@1': 25.833096037059164, 'R@10': 61.020876488685154, 'R@5': 49.44016690768524, 'R@50': 84.24027038747225}, 'per_category': {'dress': {'R@1': 22.073578595317723, 'R@10': 54.515050167224075, 'R@5': 43.97993311036789, 'R@50': 79.76588628762542}, 'shirt': {'R@1': 27.545909849749584, 'R@10': 61.769616026711184, 'R@5': 51.41903171953256, 'R@50': 84.47412353923205}, 'toptee': {'R@1': 27.879799666110184, 'R@10': 66.7779632721202, 'R@5': 52.92153589315526, 'R@50': 88.48080133555926}}, 'query_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'scope': 'full locked development validation; NOT official-val', 'search': 'exact L2 cosine FP32, reference retained, exact ties original gallery_index ascending', 'unit': 'percent'}
TRIAL_SOURCE_CONFIG = {'preparation_root': '/content/drive/MyDrive/ĐATN/data/processed/benchmark_preparation/FashionIQ/fashioniq_prep_v1_20261103', 'development_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_development/FashionIQ/tme_dev_v1_seed42', 'output_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_trial/FashionIQ', 'experiment_id': 'adapter_trial_v1_seed42', 'seed': 42, 'subset_size': 2048, 'max_epochs': 3, 'batch_size': 32, 'profile_batches': [32, 64], 'encode_batch_size': 16, 'eval_batch_size': 32, 'cache_shard_size': 128, 'adapter_width': 128, 'adapter_heads': 4, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'retrieval_temperature': 0.07, 'alignment_weight': 0.1, 'gradient_clip': 1.0, 'checkpoint_every_steps': 16, 'resume': True, 'amp': False, 'tf32': False, 'require_cuda': True, 'baseline_max_metric_deviation_pp': 0.01, 'gate_equivalence_atol': 1e-06, 'local_root': '/content/visfit_tme_adapter_trial', 'vision_dim': 768}
TRIAL_ARTIFACT_LOCKS = {'trial_request.json': '6a1f9821eed565029d586f9430b9075d2da0d1b92d8ea1d1e3402924e21b1512', 'locked_training_config.json': 'ac76a3a791c18fabb179ec0e1cd1f5255e4bf9177aa880864e3ff5bc8804fdbe', 'source_provenance.json': '6bab98312441ab955ec743ff641edea9e832546e54af17267e1aa21d736d3d33', 'cache_audit.json': 'd98f3b00094a4597186b76a3b757bdb909151983bf4bbd5d47f52030a73254ac', 'baseline/verification.json': 'acb1180f158778e7493b277119a39a0328c22468e4dd960c59063b830ebafdf5', 'retrieval_only/config.json': '38d219267c378f06bbd473f587e3b1fa6d9ba9f553e74de612e8322026d224f9', 'retrieval_only/best.json': '3b0acf4ff206ee56a21e8e23b3535feb37c04444c8d064c041febcd4c1971c41', 'retrieval_only/status.json': '25cae830703b0a577ac406393fad91c65b12507c5385fba2197866427aaff897', 'retrieval_only/metric_history.json': 'd46c1f80ea500fe08452dab961380347cc53415c5c9ff64e3886860e700e5c93', 'retrieval_only/epoch_03_per_query.csv': 'd016bcd7617eac355565d7a9785ff9da34d868e2e4080fe8d361e6ea1b7bea29', 'cache/ccb2e27db7a9f9287023105e4a6cd293f95880c1174ef37cc0011283e9e4415b/index.json': '5f01cb1aab1fea6b4fa4dd3098f7c3fee2deb50beaaedac12b9d177f071417b5'}
TRIAL_CACHE_FINGERPRINT = 'ccb2e27db7a9f9287023105e4a6cd293f95880c1174ef37cc0011283e9e4415b'
TRIAL_CODE_SHA256 = '17ae2f4fe46c69804c8fa405751a069ffc4cbe99a1ff9498cdff01aea1382a3a'
TRIAL_EXPECTED_PARAMETERS = 446978
TRIAL_EXPECTED_METRICS = {'Mean': 72.96474242652656, 'gallery_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'n_query': 1796, 'overall': {'R@1': 26.38995129377651, 'R@10': 61.41115906667187, 'R@5': 49.60673958641586, 'R@50': 84.51832578638125}, 'per_category': {'dress': {'R@1': 22.74247491638796, 'R@10': 55.852842809364546, 'R@5': 43.31103678929766, 'R@50': 79.4314381270903}, 'shirt': {'R@1': 28.547579298831387, 'R@10': 61.93656093489148, 'R@5': 52.08681135225376, 'R@50': 84.97495826377296}, 'toptee': {'R@1': 27.879799666110184, 'R@10': 66.4440734557596, 'R@5': 53.42237061769616, 'R@50': 89.14858096828047}}, 'query_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'scope': 'fixed full development validation only', 'unit': 'percent'}
TRIAL_EXPECTED_MARKER = {'phase': 'retrieval_only', 'sha256': '9a5ed5dd19eac6566e84258448f54544ea7ccc1b5ff3d3801dcd3fd1e2d2eb44', 'signature': '9102a67cc4e1fea03c15c71c937db2bb2e188f3a7f20fe575a617c948815dd82', 'step': 192}
TRIAL_DIAGNOSTIC_SEMANTIC_LOCKS = {'baseline/verification.json': {'excluded_fields': ['elapsed_seconds'], 'semantic_sha256': '3f53d48b5d5dfe81780a8153630337794689e12f22fae0b211678e8accd981ec'}}

CODE_SHA256 = '63c5666a61a6fb4afbee2b84f49dbc3d77fa8a2ee14bc47e91f34afd23674dd2'
EXTRACTOR_SHA256 = '82a37f5162dfd31ac5bbdd37bc903cf260ce8983c4788331399620274b1cac7b'


In [ ]:
# common.py — code nhúng
import os, sys, json, hashlib, math, random, time, copy, traceback, uuid, shutil
import platform, tempfile, contextlib, importlib.metadata, subprocess
from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image
from transformers import CLIPModel, CLIPTokenizer, CLIPImageProcessor

CATEGORIES = ('dress', 'shirt', 'toptee')
CAPTION_POLICY = 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1'
IMPLEMENTATION = 'visfit-tme-adapter-development-v1.0.2-storage-fixture-fix'

def require(condition, message):
    if not condition: raise RuntimeError(message)

def canonical(x):
    return json.dumps(x, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()

def fingerprint(x): return hashlib.sha256(canonical(x)).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text())

def write_bytes(path, raw, immutable=False):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if immutable and path.exists():
        require(path.read_bytes() == raw, 'Existing immutable artifact differs: '+str(path)); return
    tmp = path.with_name(path.name+'.'+uuid.uuid4().hex+'.partial')
    tmp.write_bytes(raw); os.replace(tmp, path)

def write_json(path, value, immutable=False):
    write_bytes(path, canonical(value)+b'\n', immutable)

def write_csv(path, frame, immutable=False):
    write_bytes(path, frame.to_csv(index=False, lineterminator='\n').encode(), immutable)

def safe_child(root, relative):
    root = Path(root).resolve(); path = (root/relative).resolve()
    require(path != root and root in path.parents, 'Path escapes root'); return path

def tensor_hash(module):
    h = hashlib.sha256()
    for name, value in sorted(module.state_dict().items()):
        h.update(name.encode()); h.update(str(value.dtype).encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def finite(x, name):
    require(bool(torch.isfinite(x).all()), 'NaN/Inf: '+name); return x

def unit(x):
    finite(x, 'unnormalized vector')
    require(bool((x.float().norm(dim=-1)>1e-8).all()), 'Zero retrieval vector')
    return F.normalize(x.float(), dim=-1)

def normalize_caption(x): return x.strip('.?, ')

def caption_text(a, b): return normalize_caption(a).capitalize()+' and '+normalize_caption(b)

def assess_captions(caps):
    if not isinstance(caps, list) or len(caps)!=2 or not all(isinstance(x,str) for x in caps):
        return 'schema_error', None
    content = [bool(normalize_caption(x)) and not normalize_caption(x).isspace() for x in caps]
    if not any(content): return 'both_empty_captions', None
    return None, 'single_empty_caption' if not all(content) else None

def stratified_sample(frame, n, seed):
    require(type(n) is int and 3<=n<=len(frame), 'Sample budget outside available range; change CFG explicitly')
    require(frame.query_id.is_unique, 'Duplicate query IDs')
    # Sort by query ID before RNG: stable if the input CSV order changes.
    rng = np.random.default_rng(seed); chosen=[]
    quotas = {c:n//3+int(i<n%3) for i,c in enumerate(CATEGORIES)}
    for c in CATEGORIES:
        ids=sorted(frame.loc[frame.category==c,'query_id'])
        require(len(ids)>=quotas[c], 'Insufficient category '+c+'; no automatic redistribution')
        chosen.extend(rng.choice(ids, size=quotas[c], replace=False).tolist())
    return frame.set_index('query_id',drop=False).loc[sorted(chosen)].reset_index(drop=True)

def runtime_contract():
    return {'python':platform.python_version(), 'packages':{n:importlib.metadata.version(n) for n in
        ('torch','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow')},
        'device':str(DEVICE), 'device_name':torch.cuda.get_device_name() if DEVICE.type=='cuda' else platform.machine(),
        'cuda':torch.version.cuda, 'dtype':'float32', 'amp':False, 'tf32':False, 'attention_backend':'eager'}

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def restore_rng(s):
    random.setstate(s['python']); np.random.set_state(s['numpy']); torch.set_rng_state(s['torch'])
    if s['cuda'] is not None:
        require(torch.cuda.is_available(), 'Resume CUDA checkpoint on CPU refused')
        torch.cuda.set_rng_state_all(s['cuda'])

def status(phase, **extra):
    write_json(RUN/'pilot_status.json', {'phase':phase,'implementation':IMPLEMENTATION,
        'official_val_used':False,'full_experiment_started':False,'updated_utc':pd.Timestamp.now(tz='UTC').isoformat(),**extra})

@contextlib.contextmanager
def stage(name):
    try: yield
    except BaseException as exc:
        if isinstance(exc, KeyboardInterrupt): kind='interrupted'
        else: kind='blocked'
        # No credential/download command logs are included in exception messages.
        write_bytes(RUN/(name+'_traceback.txt'), traceback.format_exc().encode())
        status(kind, stage=name, blocker=str(exc), traceback_file=name+'_traceback.txt')
        raise



In [ ]:
# checkpoint_architecture.py — code nhúng
class ReferenceInjection(nn.Module):
    def __init__(self, text_dim, vision_dim, heads, mlp):
        super().__init__()
        self.q_ln=nn.LayerNorm(text_dim); self.v_ln=nn.LayerNorm(vision_dim)
        self.v_proj=nn.Linear(vision_dim,text_dim)
        self.ca=nn.MultiheadAttention(text_dim,heads,batch_first=True,dropout=0.0)
        self.out_ln=nn.LayerNorm(text_dim); self.mlp=copy.deepcopy(mlp)
    def forward(self, h, reference):
        v=self.v_proj(self.v_ln(reference))
        z=h+self.ca(self.q_ln(h),v,v,need_weights=False)[0]
        return z+self.mlp(self.out_ln(z))

def eos_positions(ids, mask, eos_token_id):
    require(ids.shape==mask.shape and bool(((mask==0)|(mask==1)).all()), 'Token/mask shape or values invalid')
    lengths=mask.sum(-1)
    require(bool((lengths>=2).all()), 'Empty text token sequence')
    expected=torch.arange(ids.shape[1],device=ids.device)[None,:]<lengths[:,None]
    require(torch.equal(expected,mask.bool()), 'Tokenizer must right-pad contiguous sequences')
    # pad_token == EOS for CLIP; intersect with attention_mask and select first real EOS.
    is_eos=(ids==eos_token_id)&mask.bool()
    require(bool(is_eos.any(-1).all()), 'EOS absent from unpadded tokens')
    pos=is_eos.long().argmax(-1)
    require(bool((pos==lengths-1).all()), 'EOS must close each untruncated modifier')
    return pos

def text_masks(mask, dtype):
    b,n=mask.shape; neg=torch.finfo(dtype).min
    padding=torch.zeros((b,1,n,n),device=mask.device,dtype=dtype).masked_fill(~mask[:,None,None,:].bool(),neg)
    causal=torch.full((n,n),neg,dtype=dtype,device=mask.device).triu(1)
    return padding,causal[None,None].expand(b,1,n,n)

class TMEPilot(nn.Module):
    """Paper-inspired layer-level tri-branch; not the unpublished original TME."""
    def __init__(self, base, eos_token_id, active_layers):
        super().__init__()
        self.text=copy.deepcopy(base.text_model)
        self.projection=copy.deepcopy(base.text_projection)
        self.text.requires_grad_(True);self.projection.requires_grad_(True)
        self.eos_token_id=int(eos_token_id); self.active_layers=tuple(active_layers)
        self.align=nn.ModuleDict(); self.inject=nn.ModuleDict(); self.merge=nn.ModuleDict()
        self.align_norm=nn.ModuleDict(); self.align_projection=nn.ModuleDict()
        self.text.embeddings.requires_grad_(False)  # pilot choice: freeze token/position tables
        tc=base.config.text_config; vc=base.config.vision_config
        require(len(self.text.encoder.layers)==vc.num_hidden_layers, 'This implementation requires paired text/vision depths')
        require(self.active_layers and len(set(self.active_layers))==len(self.active_layers) and
                all(0<=i<len(self.text.encoder.layers) for i in self.active_layers),'Invalid active layers')
        for i in self.active_layers:
            k=str(i); block=self.text.encoder.layers[i]
            self.align[k]=copy.deepcopy(block)
            self.inject[k]=ReferenceInjection(tc.hidden_size,vc.hidden_size,tc.num_attention_heads,block.mlp)
            self.merge[k]=nn.LayerNorm(tc.hidden_size)
            self.align_norm[k]=copy.deepcopy(self.text.final_layer_norm)
            self.align_projection[k]=copy.deepcopy(base.text_projection)
            self.align_projection[k].requires_grad_(True)

    def forward(self, reference, modifier, return_alignment=False, conventional_only=False):
        # reference: layer_tokens[B,L,Nv,Dv], pooled[B,D]; contains NO target.
        ids=modifier['input_ids']; mask=modifier['attention_mask']
        pos=eos_positions(ids,mask,self.eos_token_id); b=ids.shape[0]
        h=self.text.embeddings(input_ids=ids)
        padding,causal=text_masks(mask,h.dtype); aligned=[]
        valid=mask.unsqueeze(-1).to(h.dtype)
        for i,block in enumerate(self.text.encoder.layers):
            previous=h
            conventional=block(previous,attention_mask=padding,causal_attention_mask=causal)[0]
            if i in self.active_layers and not conventional_only:
                k=str(i)
                ri=self.inject[k](previous,reference['layer_tokens'][:,self.active_layers.index(i)])
                ta=self.align[k](previous,attention_mask=padding,causal_attention_mask=causal)[0]
                # Equation 4: enhanced stream is propagated into every subsequent layer.
                h=self.merge[k](conventional+ri+ta)*valid
                a=self.align_norm[k](ta)[torch.arange(b,device=h.device),pos]
                aligned.append(self.align_projection[k](a))
            else: h=conventional*valid
        h=self.text.final_layer_norm(h)
        pooled=h[torch.arange(b,device=h.device),pos]
        text=unit(self.projection(pooled))
        q=unit(text+unit(reference['pooled']))
        if return_alignment:
            require(aligned, 'No TA outputs in conventional-only path')
            return q, torch.stack(aligned,dim=1)
        return q

    def encode_query(self, reference, modifier):
        return self(reference, modifier)  # deliberately no target argument



In [ ]:
# data_base.py — code nhúng
def inspect_sources():
    prep=Path(CFG['preparation_root']);dev=Path(CFG['development_root'])
    for name,sha in PREPARATION_ANCHORS.items():require(file_hash(prep/name)==sha,'Preparation changed: '+name)
    for name,sha in SOURCE_LOCKS.items():
        root=dev/'conventional' if name in ('config.json','best.json','metric_history.json','epoch_03_per_query.csv') else dev
        require(file_hash(root/name)==sha,'Conventional provenance changed: '+name)
    contract=read_json(prep/'extraction_contract.json');meta=read_json(prep/'feature_metadata.json')
    require(meta['contract']==contract and meta['status']=='complete' and not meta['legacy_arrays_mixed'],'Incomplete/mixed preparation')
    require(contract['caption_policy']==CAPTION_POLICY and contract['dtype']=='float32' and not contract['autocast'] and not contract['tf32'],'Extraction/caption mismatch')
    require(file_hash(prep/'feature_audit.json')==meta['feature_audit_sha256'],'Preparation feature audit checksum')
    dev_manifests=read_json(dev/'pilot_manifest_hashes.json')
    require(dev_manifests['binding']['feature_fingerprint']==meta['fingerprint'] and dev_manifests['binding']['preparation_anchors']==PREPARATION_ANCHORS,'Conventional was trained with different preparation')
    data_audit=read_json(prep/'data_audit.json');require(data_audit.get('schema_error_count')==0 and data_audit.get('both_empty_captions_count')==0 and not data_audit.get('query_or_gallery_errors'),'Preparation audit did not pass')
    frames={};warnings=[];plan=read_json(prep/'training_plan.json')
    for name in ('development_train','development_val','development_gallery'):
        f=pd.read_csv(prep/(name+'.csv'),dtype=str,keep_default_na=False)
        vr=meta['feature_views'][name];view=pd.read_csv(prep/vr['path'],dtype=str,keep_default_na=False)
        require(file_hash(prep/vr['path'])==vr['sha256'] and f.equals(view[f.columns]) and len(f)==vr['rows'],'Feature view mapping changed')
        if 'query_id' in f:
            require(f.query_id.is_unique and set(f.official_split)=={'train'},'Only official-train derived development queries')
            for row in f.itertuples():
                require(row.source_file==f'cap.{row.category}.train.json' and row.query_id==f'{row.category}:train:{int(row.source_row):06d}','Query/source mismatch')
                error,warning=assess_captions([row.caption_1,row.caption_2]);require(error is None,'Caption schema/content: '+row.query_id)
                require(row.modification_text==caption_text(row.caption_1,row.caption_2),'Caption policy changed')
                require(row.triplet_key_sha256==fingerprint([row.category,row.reference_image_id,row.target_image_id,row.caption_1,row.caption_2]),'Triplet identity mismatch')
                if warning:warnings.append({'query_id':row.query_id,'reason':warning})
        else:
            f.gallery_index=f.gallery_index.astype(int)
            for c in CATEGORIES:
                g=f[f.category==c];require(g.image_id.is_unique and g.gallery_index.tolist()==list(range(len(g))),'Gallery order mismatch')
                require(fingerprint(g.image_id.tolist())==plan['development']['gallery_id_order_sha256'][c],'Gallery lock changed')
        frames[name]=f
    train,val,gallery=[frames[k] for k in ('development_train','development_val','development_gallery')]
    require(len(train)==16204 and len(val)==1796 and len(gallery)==1796,'Locked counts changed')
    require(set(train.query_id).isdisjoint(val.query_id) and set(train.target_image_id).isdisjoint(val.target_image_id),'Development leakage')
    for c in CATEGORIES:require(set(val[val.category==c].target_image_id)<=set(gallery[gallery.category==c].image_id),'Validation target coverage')
    training=train.copy()
    for name,f in [('train',training),('validation',val),('gallery',gallery)]:write_csv(RUN/'manifests'/(name+'.csv'),f,immutable=True)
    require(file_hash(RUN/'manifests'/'validation.csv')==dev_manifests['manifests']['pilot_val']['sha256'] and file_hash(RUN/'manifests'/'gallery.csv')==dev_manifests['manifests']['development_gallery']['sha256'],'Validation/gallery differ from conventional selection protocol')
    require(file_hash(RUN/'manifests'/'train.csv')==dev_manifests['manifests']['pilot_train']['sha256'],'Full train manifest differs from locked development')
    write_json(RUN/'caption_warnings.json',warnings)
    source={'prep':prep,'dev':dev,'meta':meta,'contract':contract,'train':training,'validation':val,'gallery':gallery,
        'train_sha256':file_hash(RUN/'manifests'/'train.csv'),'validation_sha256':file_hash(RUN/'manifests'/'validation.csv'),
        'gallery_sha256':file_hash(RUN/'manifests'/'gallery.csv'),'mapping_hashes':dev_manifests['binding']['mapping_hashes']}
    write_json(RUN/'source_provenance.json',{'source_locks':SOURCE_LOCKS,'prep_anchors':PREPARATION_ANCHORS,'base_marker':EXPECTED_BASE_MARKER,
        'train_sha256':source['train_sha256'],'feature_fingerprint':meta['fingerprint'],'counts':{k:len(source[k]) for k in ('train','validation','gallery')}},immutable=True)
    return source

class FrozenConventional(nn.Module):
    def __init__(self,text,projection,eos_id):
        super().__init__();self.text=text;self.projection=projection;self.eos_id=eos_id;self.requires_grad_(False);self.eval()
    def train(self,mode=True):return super().train(False)
    def encode_text(self,modifier):
        ids,mask=modifier['input_ids'],modifier['attention_mask'];eos=eos_positions(ids,mask,self.eos_id)
        h=self.text.embeddings(input_ids=ids);padding,causal=text_masks(mask,h.dtype);valid=mask.unsqueeze(-1).to(h.dtype)
        for block in self.text.encoder.layers:h=block(h,attention_mask=padding,causal_attention_mask=causal)[0]*valid
        return self.text.final_layer_norm(h)*valid,mask,eos
    def encode_query(self,reference,modifier):
        h,mask,eos=self.encode_text(modifier);v=self.projection(h[torch.arange(len(h),device=h.device),eos])
        return unit(unit(v)+unit(reference['pooled']))

def _load_frozen_checkpoint(source):
    c=source['contract'];dev=source['dev'];record=read_json(dev/'conventional'/'config.json');marker=read_json(dev/'conventional'/'best.json')
    require(marker==EXPECTED_BASE_MARKER and marker['phase']=='conventional' and record['variant']=='conventional','Wrong base checkpoint')
    path=dev/'conventional'/'best.pt';require(file_hash(path)==marker['sha256'],'Base checkpoint SHA mismatch')
    require(c['encoder_id']=='patrickjohncyh/fashion-clip' and c['revision']=='7e3ba62ce16b379a1ab479346b66f192e76f51b7','Unexpected backbone revision')
    tokenizer=CLIPTokenizer.from_pretrained(c['encoder_id'],revision=c['revision'])
    processor=CLIPImageProcessor.from_pretrained(c['encoder_id'],revision=c['revision'])
    require(fingerprint(tokenizer.get_vocab())==c['tokenizer']['vocab_sha256'] and tokenizer.special_tokens_map==c['tokenizer']['special_tokens'] and tokenizer.padding_side=='right','Tokenizer mismatch')
    require(all(processor.to_dict().get(k)==v for k,v in c['processor'].items() if k not in ('_processor_class','image_processor_type')),'Preprocessor mismatch')
    base=CLIPModel.from_pretrained(c['encoder_id'],revision=c['revision'],dtype=torch.float32,attn_implementation='eager')
    require(tensor_hash(base)==c['encoder_weight_sha256'],'Pretrained weight hash mismatch');base.requires_grad_(False);base.eval();base.to(DEVICE)
    state=torch.load(path,map_location='cpu',weights_only=False) # own hash-bound stored checkpoint; never arbitrary pickle
    required={'signature','phase','step','epoch','cursor','model','optimizer','rng','config','cache_fingerprint','metrics','best_epoch','finished'}
    require(required<=state.keys() and state['signature']==record['signature']==marker['signature'] and state['phase']=='conventional','Checkpoint payload schema/signature mismatch')
    require(state['step']==12153 and state['epoch']==3 and state['best_epoch']==3 and state['finished'] and state['config']==record['config'],'Checkpoint selection/config changed')
    require(state['cache_fingerprint']==read_json(dev/'cache_binding.json')['fingerprint'],'Development cache provenance mismatch')
    wrapper=TMEPilot(base,tokenizer.eos_token_id,list(range(base.config.text_config.num_hidden_layers)))
    wrapper.load_state_dict(state['model'],strict=True);wrapper.requires_grad_(False);wrapper.eval();wrapper.to(DEVICE)
    frozen=FrozenConventional(wrapper.text,wrapper.projection,tokenizer.eos_token_id).to(DEVICE)
    # Small genuine encoder forward and strict schema parity, before larger feature reads.
    saved=rng_state()
    try:
        mod=tokenizer(['Red and longer',' and blue'],padding='max_length',max_length=77,return_tensors='pt');mod={k:v.to(DEVICE) for k,v in mod.items()}
        image=Image.fromarray(np.arange(224*224*3,dtype=np.uint32).reshape(224,224,3).astype(np.uint8))
        pixels=processor(images=[image,image],return_tensors='pt')['pixel_values'].to(DEVICE,dtype=torch.float32)
        with torch.inference_mode():
            out=base.vision_model(pixel_values=pixels);ref={'pooled':unit(base.visual_projection(out.pooler_output))}
            q0=wrapper(ref,mod,conventional_only=True);q1=frozen.encode_query(ref,mod)
            native=unit(unit(frozen.projection(frozen.text(**mod).pooler_output))+unit(ref['pooled']))
            require(torch.equal(q0,q1),'Extracted conventional differs from strict checkpoint architecture')
            require(torch.allclose(q1,native,atol=3e-5,rtol=3e-5),'Native trained text/adapter parity failed')
            finite(out.last_hidden_state,'vision');finite(q1,'query');require(q1.shape==(2,512),'Query shape')
    finally:restore_rng(saved)
    hashes={'vision':tensor_hash(base.vision_model),'visual_projection':tensor_hash(base.visual_projection),'text':tensor_hash(frozen)}
    write_json(RUN/'base_preflight.json',{'passed':True,'strict_load':True,'checkpoint_sha256':marker['sha256'],'schema_keys':sorted(state.keys()),
        'extracted_conventional_bitwise_parity':True,'native_forward_parity':True,'hashes':hashes,'checkpoint_epoch':state['epoch'],'device':str(DEVICE),
        'module_paths':{k:getattr(sys.modules.get(k),'__file__',None) for k in ('torch','torchvision','transformers','tokenizers','huggingface_hub')},
        'packages':{k:importlib.metadata.version(k) for k in ('torch','torchvision','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow')}})
    del state,wrapper
    return base,frozen,tokenizer,processor,hashes

def assert_backbones_frozen(base,frozen,hashes,check_hash=True):
    require(not base.training and not frozen.training and all(not p.requires_grad and p.grad is None for m in (base,frozen) for p in m.parameters()),'Frozen backbone mode/grad violation')
    if check_hash:require({'vision':tensor_hash(base.vision_model),'visual_projection':tensor_hash(base.visual_projection),'text':tensor_hash(frozen)}==hashes,'Frozen backbone weights changed')


def load_frozen_checkpoint(source):
    saved=rng_state()
    try:
        record=read_json(source['dev']/'conventional'/'config.json')
        require(record['config']['seed']==CFG['base_seed']==42,'Wrong base encoder seed')
        return _load_frozen_checkpoint(source)
    finally:restore_rng(saved)


In [ ]:
# adapter.py — code nhúng
class ResidualAdapter(nn.Module):
    def __init__(self,projection_weight,text_dim=512,vision_dim=768,width=128,heads=4):
        super().__init__();require(width%heads==0,'Adapter width/head mismatch')
        self.register_buffer('projection_weight',projection_weight.detach().clone())
        self.text_norm=nn.LayerNorm(text_dim);self.text_down=nn.Linear(text_dim,width)
        self.reference_norm=nn.LayerNorm(vision_dim);self.reference_down=nn.Linear(vision_dim,width)
        self.cross_attention=nn.MultiheadAttention(width,heads,dropout=0.,batch_first=True)
        self.ri_mlp=nn.Sequential(nn.Linear(width,width),nn.GELU(),nn.Linear(width,text_dim))
        self.ta_mlp=nn.Sequential(nn.Linear(text_dim,width),nn.GELU(),nn.Linear(width,text_dim))
        self.gate_ri=nn.Parameter(torch.zeros(()));self.gate_ta=nn.Parameter(torch.zeros(()))
    def forward(self,reference_features,text_features,mask,eos,return_aux=False,gates_off=False):
        require(text_features.ndim==3 and mask.shape==text_features.shape[:2] and eos.shape==(len(text_features),),'Adapter input shape')
        require(bool(((mask==0)|(mask==1)).all()) and bool((eos==mask.sum(-1)-1).all()),'Mask/EOS invalid')
        require(bool((eos>=1).all()) and bool((mask.sum(-1)<=mask.shape[1]).all()),'Invalid sequence length')
        require(torch.equal(mask.bool(),torch.arange(mask.shape[1],device=mask.device)[None,:]<mask.sum(-1)[:,None]),'Mask must be contiguous right padding')
        valid=mask.unsqueeze(-1).to(text_features.dtype)
        x=self.text_norm(text_features);q=self.text_down(x);kv=self.reference_down(self.reference_norm(reference_features['tokens']))
        ri=self.cross_attention(q,kv,kv,need_weights=False)[0];ri=self.ri_mlp(ri)*valid
        ta=self.ta_mlp(x)*valid
        gri=self.gate_ri.new_zeros(()) if gates_off else torch.tanh(self.gate_ri)
        gta=self.gate_ta.new_zeros(()) if gates_off else torch.tanh(self.gate_ta)
        enhanced=(text_features+gri*ri+gta*ta)*valid # no LN/MLP on the base stream
        index=torch.arange(len(eos),device=eos.device)
        text=unit(F.linear(enhanced[index,eos],self.projection_weight))
        query=unit(text+unit(reference_features['pooled'])) # reference composed exactly once
        if return_aux:
            student=F.linear((text_features+ta)[index,eos],self.projection_weight)
            return query,student,enhanced
        return query
    def encode_query(self,reference_features,text_features,mask,eos):
        return self(reference_features,text_features,mask,eos)
    def enhanced_text_sequence(self,reference_features,text_features,mask,eos):
        _,_,h=self(reference_features,text_features,mask,eos,return_aux=True)
        return h,mask,eos

def positive_mask(target_ids,device):
    require(target_ids and all(isinstance(v,str) and v for v in target_ids),'Invalid target IDs')
    return torch.tensor([[a==b for b in target_ids] for a in target_ids],dtype=torch.bool,device=device)

def retrieval_loss(query,target,target_ids,temperature):
    require(temperature>0 and query.shape==target.shape,'InfoNCE shape/temperature')
    logits=unit(query)@unit(target.detach()).T/temperature;p=positive_mask(target_ids,query.device)
    def direction(x,m):return -((F.log_softmax(x,dim=1)*m).sum(1)/m.sum(1)).mean()
    return (direction(logits,p)+direction(logits.T,p.T))/2

def objective(model,batch):
    query=model.encode_query(batch['reference_features'],batch['text_features'],batch['mask'],batch['eos'])
    loss=retrieval_loss(query,batch['target'],batch['target_ids'],CFG['retrieval_temperature'])
    finite(loss,'retrieval loss');return loss

def adapter_initial(projection_weight):
    saved=rng_state();seed_all(CFG['adapter_seed'])
    try:return ResidualAdapter(projection_weight,text_dim=projection_weight.shape[1],vision_dim=CFG.get('vision_dim',768),width=CFG['adapter_width'],heads=CFG['adapter_heads']).float().to(DEVICE)
    finally:restore_rng(saved)

def gradient_summary(model):
    groups={'gate':[model.gate_ri,model.gate_ta],
        'RI':list(model.text_down.parameters())+list(model.reference_norm.parameters())+list(model.reference_down.parameters())+list(model.cross_attention.parameters())+list(model.ri_mlp.parameters()),
        'TA':list(model.ta_mlp.parameters()),'shared_norm':list(model.text_norm.parameters())}
    out=[]
    for params in groups.values():out.append(torch.stack([p.grad.detach().float().square().sum() for p in params if p.grad is not None]).sum().sqrt() if any(p.grad is not None for p in params) else model.gate_ri.new_zeros(()))
    values=torch.stack(out).detach().cpu().tolist();require(all(math.isfinite(x) for x in values),'Nonfinite branch gradient')
    return dict(zip(groups,values))



In [ ]:
# ranking.py — code nhúng
def exact_ranks(query,gallery,target_positions):
    q=np.asarray(query,dtype=np.float32); g=np.asarray(gallery,dtype=np.float32)
    require(q.ndim==g.ndim==2 and q.shape[1]==g.shape[1] and len(q)>0 and len(g)>0,'Ranking shapes invalid')
    require(np.isfinite(q).all() and np.isfinite(g).all(),'Nonfinite ranking input')
    require((np.linalg.norm(q,axis=-1)>1e-8).all() and (np.linalg.norm(g,axis=-1)>1e-8).all(),'Zero ranking input')
    q=q/np.linalg.norm(q,axis=-1,keepdims=True);g=g/np.linalg.norm(g,axis=-1,keepdims=True)
    targets=np.asarray(target_positions,dtype=np.int64)
    require(targets.shape==(len(q),) and (targets>=0).all() and (targets<len(g)).all(),'Target gallery alignment invalid')
    scores=q@g.T; ts=scores[np.arange(len(q)),targets]
    ranks=1+(scores>ts[:,None]).sum(1)+((scores==ts[:,None])&(np.arange(len(g))[None,:]<targets[:,None])).sum(1)
    return ranks

def macro_metrics(per_category):
    require(set(per_category)==set(CATEGORIES),'Macro requires three categories')
    overall={f'R@{k}':float(np.mean([per_category[c][f'R@{k}'] for c in CATEGORIES])) for k in (1,5,10,50)}
    return {'overall':overall,'Mean':(overall['R@10']+overall['R@50'])/2}

def pure_tests():
    cases=[(['red','long'],None,None,'Red and long'),(['','long'],None,'single_empty_caption',' and long'),
           (['red',''],None,'single_empty_caption','Red and '),([' .?, ','long'],None,'single_empty_caption',' and long'),
           ([' \t ','.?, '],'both_empty_captions',None,None),(['',''],'both_empty_captions',None,None),
           (['red',1],'schema_error',None,None),(['red'],'schema_error',None,None),('red','schema_error',None,None)]
    for caps,error,warning,text in cases:
        require(assess_captions(caps)==(error,warning),'Caption assessment mismatch')
        if text is not None:require(caption_text(*caps)==text,'Caption policy mismatch')
    require(caption_text(' .Red?, ',' BLUE. ')=='Red and BLUE','Caption normalization strips only .?, and ASCII space at both ends')
    require(normalize_caption('\tblue\t')=='\tblue\t','Tabs must not be silently stripped by policy')
    q=np.array([[1,0],[0,1]],np.float32);g=np.array([[1,0],[1,0],[0,1]],np.float32)
    require(exact_ranks(q,g,[1,2]).tolist()==[2,1],'Manual rank/tie test failed')
    rng=np.random.default_rng(42);a=rng.normal(size=(7,8)).astype(np.float32);b=rng.normal(size=(13,8)).astype(np.float32);targets=np.arange(7)
    a/=np.linalg.norm(a,axis=-1,keepdims=True);b/=np.linalg.norm(b,axis=-1,keepdims=True)
    orders=np.argsort(-(a@b.T),axis=1,kind='stable')
    expected=[int(np.flatnonzero(row==t)[0])+1 for row,t in zip(orders,targets)]
    require(exact_ranks(a,b,targets).tolist()==expected,'Exact rank and stable sorting disagree')
    cat={c:{'R@1':0.,'R@5':10.,'R@10':20.+i*10,'R@50':60.+i*10} for i,c in enumerate(CATEGORIES)}
    require(macro_metrics(cat)['overall']['R@10']==30 and macro_metrics(cat)['Mean']==50,'Macro/Mean test failed')
    ids=['a','a','b','c'];mask=positive_mask(ids,torch.device('cpu'));require(mask[0,1] and not mask[0,2],'Duplicate target positive mask failed')
    x=unit(torch.arange(1,17,dtype=torch.float32).reshape(4,4));y=x.flip(0)
    logits=x@y.T/0.07
    manual=[]
    for scores,p in ((logits,mask),(logits.T,mask.T)):
        manual.append(torch.stack([-F.log_softmax(scores[i],dim=0)[p[i]].mean() for i in range(4)]).mean())
    require(torch.allclose(retrieval_loss(x,y,ids,.07),sum(manual)/2),'Multi-positive reduction mismatch')
    return {'caption_cases':len(cases)+2,'exact_rank_tie_and_stable_sort':True,'macro_mean':True,'duplicate_target_multi_positive':True}


In [ ]:
# checkpoint.py — code nhúng
def checkpoint_save(path,state,signature):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    require(state['signature']==signature,'Checkpoint signature differs')
    with tempfile.NamedTemporaryFile(dir=path.parent,suffix='.pt',delete=False) as f:temp=Path(f.name)
    try:
        torch.save(state,temp);meta={'signature':signature,'sha256':file_hash(temp),'phase':state.get('phase'),'step':state.get('step')}
        write_json(path.with_suffix('.pending.json'),meta);os.replace(temp,path);write_json(path.with_suffix('.json'),meta)
    finally:
        if temp.exists():temp.unlink()

def checkpoint_load(path,signature):
    path=Path(path);marker=path.with_suffix('.json');pending=path.with_suffix('.pending.json');sha=file_hash(path)
    if pending.is_file():
        p=read_json(pending)
        if p.get('sha256')==sha:
            require(p['signature']==signature,'Pending checkpoint signature');write_json(marker,p)
    m=read_json(marker);require(m['signature']==signature and m['sha256']==sha,'Checkpoint checksum/config mismatch')
    state=torch.load(path,map_location='cpu',weights_only=False)
    require(state['signature']==signature,'Checkpoint payload identity differs');return state

def sync_checkpoint(local,destination,signature):
    destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)
    marker=read_json(Path(local).with_suffix('.json'));require(marker['signature']==signature,'Checkpoint sync signature')
    write_json(destination.with_suffix('.pending.json'),marker)
    temp=destination.with_suffix('.copying');shutil.copyfile(local,temp);require(file_hash(temp)==marker['sha256'],'Drive checkpoint sync corrupt')
    os.replace(temp,destination);write_json(destination.with_suffix('.json'),marker)

def epoch_batches(frame,batch,seed,epoch):
    ids=np.asarray(sorted(frame.query_id),dtype=object);order=np.random.default_rng(seed+epoch).permutation(ids)
    require(len(ids)>=2 and batch>=2,'Contrastive batch too small')
    batches=[order[i:i+batch].tolist() for i in range(0,len(ids),batch)]
    if len(batches)>1 and len(batches[-1])==1:batches[-2].extend(batches.pop())
    return batches



In [ ]:
# provenance.py — code nhúng
def validate_trial_locks(root):
    receipts={}
    for name,expected_sha in TRIAL_ARTIFACT_LOCKS.items():
        path=Path(root)/name
        if name in TRIAL_DIAGNOSTIC_SEMANTIC_LOCKS:
            rule=TRIAL_DIAGNOSTIC_SEMANTIC_LOCKS[name];value=read_json(path)
            require(isinstance(value,dict),'Diagnostic report schema differs: '+name)
            for field in rule['excluded_fields']:
                require(type(value.get(field)) in (int,float) and math.isfinite(value[field]) and value[field]>=0,'Invalid diagnostic timing: '+name+'/'+field)
            stable={k:v for k,v in value.items() if k not in rule['excluded_fields']}
            require(fingerprint(stable)==rule['semantic_sha256'],'Trial scientific report content changed: '+name)
            receipts[name]={'verification':'semantic','semantic_sha256':fingerprint(stable),'excluded_fields':rule['excluded_fields'],'current_file_sha256':file_hash(path)}
        else:
            require(file_hash(path)==expected_sha,'Trial source changed: '+name)
            receipts[name]={'verification':'byte_sha256','sha256':expected_sha}
    return receipts

def init_development():
    global RUN,DEVICE,LOCAL,VALIDATION_SESSION_ID
    require(CFG['base_seed']==42 and isinstance(CFG['adapter_seed'],int) and CFG['adapter_seed']>=0,'Base seed fixed42; adapter seed must be explicit')
    require(CFG['train_count']==16204 and 1<=CFG['max_epochs']<=10 and CFG['patience']>=1,'Development-only budget')
    require('alignment_weight' not in CFG,'Retrieval-only experiment has no alignment-loss setting')
    require(CFG['amp'] is False and CFG['tf32'] is False and CFG['batch_size']==32,'Locked FP32 batch32; choose new tested experiment for another batch')
    require(CFG['adapter_width']==128 and CFG['adapter_heads']==4 and CFG['vision_dim']==768,'Preserve trial adapter architecture')
    require(f'base{CFG["base_seed"]}' in CFG['experiment_id'] and f'adapter{CFG["adapter_seed"]}' in CFG['experiment_id'],'Experiment name must show base/adapter seeds')
    require(Path(CFG['experiment_id']).name==CFG['experiment_id'],'Invalid experiment name')
    DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu');require(not CFG['require_cuda'] or DEVICE.type=='cuda','Select Colab GPU')
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False;torch.backends.cudnn.benchmark=False;torch.set_default_dtype(torch.float32)
    VALIDATION_SESSION_ID=uuid.uuid4().hex
    seed_all(CFG['adapter_seed']);RUN=Path(CFG['output_root'])/CFG['experiment_id'];LOCAL=Path(CFG['local_root'])/CFG['experiment_id']
    for original in (CFG['trial_root'],CFG['development_root'],CFG['preparation_root']):
        original=Path(original);require(RUN!=original and not RUN.is_relative_to(original) and not original.is_relative_to(RUN),'Output overlaps source')
    if RUN.exists() and any(RUN.iterdir()):require(CFG['resume'],'Existing experiment requires resume=True')
    RUN.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)
    write_json(RUN/'experiment_config.json',{'config':CFG,'code_sha256':CODE_SHA256,'implementation':IMPLEMENTATION,'objective':'retrieval only; no auxiliary alignment loss'},immutable=True)
    write_json(RUN/'runtime_contract.json',runtime_contract(),immutable=True)
    write_json(RUN/'bootstrap_report.json',BOOTSTRAP)

@contextlib.contextmanager
def timed_phase(name):
    start=time.perf_counter();status='completed'
    try:yield
    except BaseException:status='failed';raise
    finally:
        row={'phase':name,'status':status,'elapsed_seconds':time.perf_counter()-start,'recorded_utc':pd.Timestamp.now(tz='UTC').isoformat()}
        path=RUN/'pipeline_timing_sessions.json';data=read_json(path) if path.exists() else []
        data.append(row);write_json(path,data)

def inspect_trial_artifacts(source,frozen):
    root=Path(CFG['trial_root'])
    require(root==Path(TRIAL_SOURCE_CONFIG['output_root'])/TRIAL_SOURCE_CONFIG['experiment_id'],'Trial root must come from source notebook configuration')
    lock_audit=validate_trial_locks(root)
    write_json(RUN/'trial_source_lock_audit.json',lock_audit)
    request=read_json(root/'trial_request.json');config=read_json(root/'retrieval_only'/'config.json');binding=config['binding']
    require(request['config']==TRIAL_SOURCE_CONFIG and request['code_sha256']==TRIAL_CODE_SHA256,'Unexpected trial configuration/code')
    require(config['signature']==fingerprint(binding) and binding['variant']=='retrieval_only' and binding['alignment_weight']==0,'Trial is not verified retrieval-only')
    require(binding['base_checkpoint']==EXPECTED_BASE_MARKER['sha256'] and binding['validation_sha256']==source['validation_sha256'] and binding['gallery_sha256']==source['gallery_sha256'],'Historical comparison uses different backbone/validation/gallery')
    subset_path=root/'manifests'/'subset.csv';require(file_hash(subset_path)==binding['subset_sha256'],'Historical subset manifest hash')
    subset=pd.read_csv(subset_path,dtype=str,keep_default_na=False);require(len(subset)==2048 and subset.query_id.is_unique,'Historical subset coverage')
    expected=source['train'].set_index('query_id',drop=False).loc[subset.query_id]
    require(expected.to_dict('records')==subset.to_dict('records'),'Historical subset no longer maps to locked full train')
    source['trial_subset']=subset
    marker=read_json(root/'retrieval_only'/'best.json');require(marker==TRIAL_EXPECTED_MARKER,'Trial best marker differs')
    path=root/'retrieval_only'/'best.pt';require(file_hash(path)==marker['sha256'],'Actual trial checkpoint hash differs')
    state=torch.load(path,map_location='cpu',weights_only=False)
    require(state['signature']==config['signature']==marker['signature'] and state['binding']==binding and state['config']==request['config'],'Actual trial checkpoint schema/provenance')
    require(state['finished'] and state['epoch']==3 and state['best_epoch']==3 and state['step']==192,'Trial budget/selection differs')
    projection=state['model']['projection_weight']
    require(hashlib.sha256(projection.numpy().tobytes()).hexdigest()==read_json(root/'locked_training_config.json')['projection_hash'] and torch.equal(projection,frozen.projection.weight.detach().cpu()),'Trial/base projection weight space differs')
    saved=rng_state()
    try:
        model=ResidualAdapter(state['model']['projection_weight'],width=128,heads=4)
        model.load_state_dict(state['model'],strict=True)
        count=sum(p.numel() for p in model.parameters() if p.requires_grad);require(count==config['trainable_parameters']==TRIAL_EXPECTED_PARAMETERS==446978,'Adapter count differs; review before training')
    finally:restore_rng(saved)
    metrics=read_json(root/'retrieval_only'/'metric_history.json');best=max(metrics,key=lambda x:(x['metrics']['Mean'],-x['epoch']))
    require(best['epoch']==3 and best['metrics']==TRIAL_EXPECTED_METRICS,'Historical trial metric selection changed')
    old=pd.read_csv(root/'retrieval_only'/'epoch_03_per_query.csv',dtype={'query_id':str})
    require(len(old)==len(source['validation']) and old.query_id.is_unique,'Trial prediction coverage')
    merged=source['validation'][['query_id','category','reference_image_id','target_image_id']].merge(old,on=['query_id','category','reference_image_id','target_image_id'],validate='one_to_one')
    require(len(merged)==len(old),'Trial prediction ID mapping differs')
    per={c:{f'R@{k}':float((merged[merged.category==c].target_rank.to_numpy()<=k).mean()*100) for k in (1,5,10,50)} for c in CATEGORIES}
    require(all(abs(per[c][k]-best['metrics']['per_category'][c][k])<1e-10 for c in CATEGORIES for k in per[c]),'Trial metrics do not match actual ranks')
    report={'passed':True,'checkpoint_sha256':marker['sha256'],'base_checkpoint_sha256':EXPECTED_BASE_MARKER['sha256'],'trial_root':str(root),
       'metrics':best['metrics'],'best_epoch':3,'adapter_training_samples':2048,'budget_epochs':3,'seed':42,'trainable_parameters':count,
       'initial_sha256':binding['initial_sha256'],'strict_load':True,'same_validation_gallery':True,
       'train_compute_seconds':state['train_compute_seconds'],'validation_seconds':state['validation_seconds'],
       'train_wall_seconds':None,'checkpoint_seconds':None,'pipeline_wall_seconds':None,
       'historical_timing_note':'Trial measured compute and validation; no total wall/checkpoint measurement, do not substitute compute for total runtime.'}
    write_json(RUN/'verified_trial_reference.json',report,immutable=True)
    del state,model
    return report

class TrialCacheReader:
    """Read-only legacy cache; never writes/rebuilds old trial markers or arrays."""
    def __init__(self,source,new_contract,new_specs):
        self.key=TRIAL_CACHE_FINGERPRINT;self.root=Path(CFG['trial_root'])/'cache'/self.key
        index_path=self.root/'index.json';require(file_hash(index_path)==TRIAL_ARTIFACT_LOCKS[f'cache/{self.key}/index.json'],'Trial cache index hash')
        index=read_json(index_path);old=index['contract'];self.specs=index['specs'];self.step=index['shard_size']
        require(index['fingerprint']==fingerprint(old)==self.key and old['code_sha256']==TRIAL_CODE_SHA256,'Unknown legacy extraction version/fingerprint')
        fields=('base_checkpoint_sha256','text','image_extraction_contract','feature_fingerprint','source_image_index_sha256',
                'text_encode_batch_size','caption_policy','dtype','attention_backend','amp','tf32','runtime')
        self.differences={k:{'trial':old.get(k),'development':new_contract.get(k)} for k in fields if old.get(k)!=new_contract.get(k)}
        # New row coverage/global code version differs by design. Only known old extractor + exact physical fields permit reuse.
        self.compatible=not self.differences
        expected_queries=set(source['trial_subset'].query_id)|set(source['validation'].query_id)
        require(set(index['specs']['text']['ids'])==expected_queries,'Historical cache query coverage differs')
        self.maps={r:{v:i for i,v in enumerate(s['ids'])} for r,s in self.specs.items()}
        require(all(len(self.maps[r])==len(s['ids']) and s['shape']==new_specs[r]['shape'] and s['dtype']==new_specs[r]['dtype'] for r,s in self.specs.items()),'Trial cache role schema/ID mismatch')
        self.local=LOCAL/'trial_cache_mirror'/self.key;self.local.mkdir(parents=True,exist_ok=True);self.loaded=OrderedDict();self.verified=set()
    def take(self,role,ids):
        require(self.compatible,'Incompatible extraction must not be reused');result=[];spec=self.specs[role]
        (self.local/role).mkdir(exist_ok=True)
        for image_id in ids:
            row=self.maps[role][image_id];start=row//self.step*self.step;end=min(start+self.step,len(spec['ids']));name=f'{start:07d}_{end:07d}'
            key=(role,start);p=self.root/role/(name+'.npy');local=self.local/role/p.name
            if key not in self.verified:
                m=read_json(p.with_suffix('.json'));expected={'fingerprint':self.key,'role':role,'ids_sha256':fingerprint(spec['ids'][start:end]),'shape':[end-start,*spec['shape']],'dtype':spec['dtype']}
                require(all(m.get(k)==v for k,v in expected.items()) and file_hash(p)==m['sha256'],'Old trial cache corrupt; preserve and investigate')
                if not local.exists() or file_hash(local)!=m['sha256']:
                    temp=local.with_suffix('.copying');shutil.copyfile(p,temp);require(file_hash(temp)==m['sha256'],'Old cache copy corrupt');os.replace(temp,local)
                a=np.load(local,mmap_mode='r',allow_pickle=False);require(list(a.shape)==expected['shape'] and str(a.dtype)==spec['dtype'] and np.isfinite(a).all(),'Old cache values invalid')
                if role in ('reference_pooled','target_pooled'):require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Old cached vectors not unit')
                self.verified.add(key)
            if key not in self.loaded:
                if len(self.loaded)>=16:self.loaded.popitem(last=False)
                self.loaded[key]=np.load(local,mmap_mode='r',allow_pickle=False)
            self.loaded.move_to_end(key);result.append(np.asarray(self.loaded[key][row-start]).copy())
        return np.stack(result)
    def combine(self,role,ids,compute,reuse):
        shared=[v for v in ids if self.compatible and v in self.maps[role]];missing=[v for v in ids if v not in set(shared)]
        values={}
        if shared:values.update(zip(shared,self.take(role,shared)))
        if missing:values.update(zip(missing,compute(missing)))
        reuse[role]=reuse.get(role,0)+len(shared)
        return np.stack([values[v] for v in ids])


In [ ]:
# cache.py — code nhúng
def full_cache_location(contract,specs):
    # A prior FULL cache may be read only when its entire extraction/manifest/index contract matches.
    # Partial trial reuse still goes through TrialCacheReader by ID below.
    remote,local=RUN/'cache',LOCAL/'cache';prior=CFG.get('verified_full_cache_source');key=fingerprint(contract)
    decision={'fingerprint':key,'reused_full_cache':False,'source':prior}
    if prior:
        old_remote=Path(prior['run_root'])/'cache';old_local=Path(prior['local_root'])/'cache';index=old_remote/key/'index.json'
        require(Path(prior['run_root'])!=RUN,'Previous-cache source must differ from output run')
        if index.is_file():
            expected={'fingerprint':key,'contract':contract,'specs':specs,'shard_size':CFG['cache_shard_size']}
            require(read_json(index)==expected,'Prior full-cache index/IDs/contract mismatch')
            for role,spec in specs.items():
                for start in range(0,len(spec['ids']),CFG['cache_shard_size']):
                    end=min(start+CFG['cache_shard_size'],len(spec['ids']));path=old_remote/key/role/f'{start:07d}_{end:07d}.npy'
                    require(path.is_file() and path.with_suffix('.json').is_file(),'Prior full cache incomplete; preserve and investigate: '+role+'/'+path.name)
            remote,local=old_remote,old_local;decision.update(reused_full_cache=True,verification='Exact index/IDs/extraction contract; every shard rechecked by TrialStore.prepare; existing source arrays are read only')
        else:decision['reason']='No full cache at the current physical fingerprint; build a separate new cache, without mixing incompatible arrays'
    decision.update(remote_cache_root=str(remote),local_cache_root=str(local))
    write_json(RUN/'full_cache_reuse.json',decision);return remote,local

class PreparationStore:
    def __init__(self,source,kind):
        require(kind in ('reference','target'),'Only final image features may be reused')
        self.kind=kind;self.root=source['prep']/'features'/source['meta']['fingerprint']/kind
        require(file_hash(self.root/'index.json')==source['meta']['store_index_sha256'][kind],'Source feature index SHA')
        self.index=read_json(self.root/'index.json');self.ids=self.index['ids'];self.mapping={v:i for i,v in enumerate(self.ids)}
        require(len(self.mapping)==len(self.ids) and self.index['fingerprint']==source['meta']['fingerprint'] and self.index['kind']==kind,'Source index identity')
        mapping_path=self.root.parent/source['meta']['mapping_files'][kind]
        require(file_hash(mapping_path)==source['mapping_hashes'][kind] and read_json(mapping_path)==self.mapping,'Source mapping differs from conventional')
        require(self.index['shape_tail']==({'reference':[50,768],'target':[512]}[kind]) and self.index['dtype']=='float32','Image feature schema')
        self.local=LOCAL/'source_images'/source['meta']['fingerprint']/kind;self.local.mkdir(parents=True,exist_ok=True)
        self.verified=set();self.loaded=OrderedDict()
    def take(self,ids):
        result=[];step=self.index['shard_size']
        for image_id in ids:
            require(image_id in self.mapping,'Image ID absent: '+image_id)
            row=self.mapping[image_id];start=row//step*step;end=min(start+step,len(self.ids));name=f'{start:07d}_{end:07d}'
            local=self.local/(name+'.npy')
            if start not in self.verified:
                marker=read_json(self.root/(name+'.json'))
                expected={'fingerprint':self.index['fingerprint'],'kind':self.kind,'ids_sha256':fingerprint(self.ids[start:end]),'shape':[end-start,*self.index['shape_tail']],'dtype':'float32'}
                require(all(marker.get(k)==v for k,v in expected.items()),'Source shard schema/ID mismatch')
                if not local.is_file() or file_hash(local)!=marker['sha256']:
                    temp=local.with_suffix('.copying');shutil.copyfile(self.root/(name+'.npy'),temp)
                    require(file_hash(temp)==marker['sha256'],'Source feature corrupt');os.replace(temp,local)
                a=np.load(local,mmap_mode='r',allow_pickle=False)
                require(list(a.shape)==expected['shape'] and a.dtype==np.float32 and np.isfinite(a).all(),'Source contents invalid')
                if self.kind=='target':require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Source target not unit')
                self.verified.add(start)
            if start not in self.loaded:
                if len(self.loaded)>=16:self.loaded.popitem(last=False)
                self.loaded[start]=np.load(local,mmap_mode='r',allow_pickle=False)
            self.loaded.move_to_end(start);result.append(np.asarray(self.loaded[start][row-start]).copy())
        return np.stack(result)

class TrialStore:
    def __init__(self,root,local,contract,specs,shard_size):
        self.key=fingerprint(contract);self.root=Path(root)/self.key;self.local=Path(local)/self.key
        self.root.mkdir(parents=True,exist_ok=True);self.local.mkdir(parents=True,exist_ok=True)
        self.specs=specs;self.contract=contract;self.step=shard_size;self.maps={k:{v:i for i,v in enumerate(s['ids'])} for k,s in specs.items()}
        require(all(len(self.maps[k])==len(s['ids']) for k,s in specs.items()),'Duplicate cached IDs')
        self.index={'fingerprint':self.key,'contract':contract,'specs':specs,'shard_size':shard_size}
        write_json(self.root/'index.json',self.index,immutable=True);self.loaded=OrderedDict();self.paths={}
    def prepare(self,role,compute):
        spec=self.specs[role];ids=spec['ids'];dtype=np.dtype(spec['dtype']);tail=spec['shape']
        (self.root/role).mkdir(exist_ok=True);(self.local/role).mkdir(exist_ok=True)
        for start in range(0,len(ids),self.step):
            end=min(start+self.step,len(ids));name=f'{start:07d}_{end:07d}';p=self.root/role/(name+'.npy');marker=p.with_suffix('.json');pending=p.with_suffix('.pending.json')
            expected={'fingerprint':self.key,'role':role,'ids_sha256':fingerprint(ids[start:end]),'shape':[end-start,*tail],'dtype':spec['dtype']}
            if not marker.is_file() and p.is_file() and pending.is_file():
                m=read_json(pending);require(all(m.get(k)==v for k,v in expected.items()) and file_hash(p)==m['sha256'],'Pending cache corrupt');write_json(marker,m,immutable=True)
            local=self.local/role/p.name
            if marker.is_file():
                m=read_json(marker);require(all(m.get(k)==v for k,v in expected.items()),'Cache fingerprint/ID/shape differs')
                require(p.is_file() and file_hash(p)==m['sha256'],'Committed Drive cache corrupt')
                if not local.is_file() or file_hash(local)!=m['sha256']:
                    temp=local.with_suffix('.copying');shutil.copyfile(p,temp);require(file_hash(temp)==m['sha256'],'Local cache copy corrupt');os.replace(temp,local)
            else:
                require(not p.exists(),'Uncommitted orphan cache; preserve and investigate')
                a=np.asarray(compute(ids[start:end]),dtype=dtype)
                require(list(a.shape)==expected['shape'] and np.isfinite(a).all(),'Computed cache shape/finite')
                temp=local.with_suffix('.partial');
                with temp.open('wb') as f:np.save(f,a,allow_pickle=False)
                m={**expected,'sha256':file_hash(temp)};write_json(pending,m)
                remote=p.with_suffix('.copying');shutil.copyfile(temp,remote);require(file_hash(remote)==m['sha256'],'Cache sync corrupt')
                os.replace(remote,p);write_json(marker,m,immutable=True);os.replace(temp,local)
            a=np.load(local,mmap_mode='r',allow_pickle=False)
            require(list(a.shape)==expected['shape'] and a.dtype==dtype and np.isfinite(a).all(),'Local cache invalid')
            if role in ('reference_pooled','target_pooled'):require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Cached embedding not unit')
            self.paths[role,start]=local
    def take(self,role,ids):
        # Training hot path: prevalidated local mmap, no Drive/JSON/checksum access.
        result=[]
        for image_id in ids:
            row=self.maps[role][image_id];start=row//self.step*self.step;key=(role,start)
            require(key in self.paths,'Run cache preflight before use')
            if key not in self.loaded:
                if len(self.loaded)>=16:self.loaded.popitem(last=False)
                self.loaded[key]=np.load(self.paths[key],mmap_mode='r',allow_pickle=False)
            self.loaded.move_to_end(key);result.append(np.asarray(self.loaded[key][row-start]).copy())
        return np.stack(result)

def prepare_cache(source,base,frozen,tokenizer):
    ref_store=PreparationStore(source,'reference');target_store=PreparationStore(source,'target')
    for role in ('train','validation'):
        view_name='development_train' if role=='train' else 'development_val'
        vr=source['meta']['feature_views'][view_name];view=pd.read_csv(source['prep']/vr['path'],dtype=str,keep_default_na=False).set_index('query_id',drop=False)
        for row in source[role].itertuples():
            v=view.loc[row.query_id]
            require(int(v.reference_feature_index)==ref_store.mapping[row.reference_image_id] and int(v.target_feature_index)==target_store.mapping[row.target_image_id],'Source row/ID alignment differs')
    queries=pd.concat([source['train'],source['validation']],ignore_index=True).sort_values('query_id')
    require(queries.query_id.is_unique,'Train/val query overlap');query_ids=queries.query_id.tolist();by_id=queries.set_index('query_id',drop=False)
    refs=sorted(set(queries.reference_image_id));targets=sorted(set(source['train'].target_image_id)|set(source['gallery'].image_id))
    contract={'implementation':'visfit-adapter-development-cache-v1','extractor_schema':'trained-conventional-finalLN-final-image-v1','extractor_source_sha256':EXTRACTOR_SHA256,'base_checkpoint_sha256':EXPECTED_BASE_MARKER['sha256'],
        'text':'strict trained conventional last stream -> final LN, padded outputs zero; frozen trained projection applied later',
        'image_extraction_contract':source['contract'],'feature_fingerprint':source['meta']['fingerprint'],
        'train_sha256':source['train_sha256'],'validation_sha256':source['validation_sha256'],'gallery_sha256':source['gallery_sha256'],
        'source_image_index_sha256':{k:source['meta']['store_index_sha256'][k] for k in ('reference','target')},
        'text_encode_batch_size':CFG['encode_batch_size'],'cache_shard_size':CFG['cache_shard_size'],
        'caption_policy':CAPTION_POLICY,'dtype':'float32','attention_backend':'eager','amp':False,'tf32':False,
        'runtime':{'packages':{k:importlib.metadata.version(k) for k in ('torch','transformers','numpy')},'device':str(DEVICE),'gpu':torch.cuda.get_device_name() if DEVICE.type=='cuda' else None}}
    specs={'text':{'ids':query_ids,'shape':[77,512],'dtype':'float32'},'mask':{'ids':query_ids,'shape':[77],'dtype':'int64'},
        'eos':{'ids':query_ids,'shape':[],'dtype':'int64'},'input_ids':{'ids':query_ids,'shape':[77],'dtype':'int64'},
        'reference':{'ids':refs,'shape':[50,768],'dtype':'float32'},'reference_pooled':{'ids':refs,'shape':[512],'dtype':'float32'},
        'target_pooled':{'ids':targets,'shape':[512],'dtype':'float32'}}
    payload=sum(len(s['ids'])*math.prod(s['shape'])*np.dtype(s['dtype']).itemsize for s in specs.values())
    # Source final-image shards are small; reserve their full image-store footprint plus checkpoint/sync temporaries.
    source_bytes=sum(len(st.ids)*math.prod(st.index['shape_tail'])*4 for st in (ref_store,target_store))
    write_json(RUN/'cache_resource_plan.json',{'adapter_cache_payload_bytes':payload,'source_image_store_upper_bound_bytes':source_bytes,
        'local_disk_free':shutil.disk_usage(LOCAL).free,'no_multilayer_cache':True})
    key=fingerprint(contract);existing=sum(p.stat().st_size for p in (LOCAL/'cache'/key).rglob('*.npy')) if (LOCAL/'cache'/key).exists() else 0
    require(shutil.disk_usage(LOCAL).free>max(0,payload-existing)+source_bytes+3*1024**3,'Insufficient local disk before cache')
    remote_cache,local_cache=full_cache_location(contract,specs)
    store=TrialStore(remote_cache,local_cache,contract,specs,CFG['cache_shard_size'])
    old=TrialCacheReader(source,contract,specs)
    reuse={}
    def text_values(ids,role):
        rows=by_id.loc[ids];values=[]
        for start in range(0,len(rows),CFG['encode_batch_size']):
            r=rows.iloc[start:start+CFG['encode_batch_size']];sentences=r.modification_text.tolist()
            lengths=[len(x) for x in tokenizer(sentences,padding=False,truncation=False)['input_ids']]
            require(max(lengths)<=77,'Overlength caption; do not truncate/drop')
            mod=tokenizer(sentences,padding='max_length',max_length=77,truncation=False,return_tensors='pt');mod={k:v.to(DEVICE) for k,v in mod.items()}
            if role=='text':
                with torch.inference_mode():value=frozen.encode_text(mod)[0]
            elif role=='eos':value=eos_positions(mod['input_ids'],mod['attention_mask'],tokenizer.eos_token_id)
            else:value=mod['attention_mask' if role=='mask' else 'input_ids']
            values.append(value.cpu().numpy())
        return np.concatenate(values)
    # Only missing text shards call the frozen encoder; mask/EOS/IDs need tokenizer alone.
    for role in ('text','mask','eos','input_ids'):store.prepare(role,lambda ids,r=role:old.combine(r,ids,lambda missing:text_values(missing,r),reuse))
    store.prepare('reference',lambda ids:old.combine('reference',ids,ref_store.take,reuse))
    def pooled_refs(ids):
        values=[]
        for start in range(0,len(ids),CFG['encode_batch_size']):
            raw=torch.from_numpy(ref_store.take(ids[start:start+CFG['encode_batch_size']])).to(DEVICE)
            with torch.inference_mode():p=unit(base.visual_projection(base.vision_model.post_layernorm(raw[:,0])))
            values.append(p.cpu().numpy())
        return np.concatenate(values)
    store.prepare('reference_pooled',lambda ids:old.combine('reference_pooled',ids,pooled_refs,reuse))
    store.prepare('target_pooled',lambda ids:old.combine('target_pooled',ids,target_store.take,reuse))
    write_json(RUN/'trial_cache_reuse.json',{'old_fingerprint':old.key,'compatible':old.compatible,'differences':old.differences,'copied_rows_this_session':reuse,'expected_shared_rows':{k:len(set(s['ids'])&set(old.specs[k]['ids'])) if old.compatible else 0 for k,s in specs.items()}})
    # Exhaustive alignment checks on every used query; this is not a sample audit.
    for start in range(0,len(query_ids),128):
        ids=query_ids[start:start+128];mask=torch.from_numpy(store.take('mask',ids));token_ids=torch.from_numpy(store.take('input_ids',ids));eos=torch.from_numpy(store.take('eos',ids))
        require(torch.equal(eos_positions(token_ids,mask,tokenizer.eos_token_id),eos),'Cached mask/EOS mapping')
        seq=store.take('text',ids);require(np.count_nonzero(seq[mask.numpy()==0])==0,'Nonzero cached padding')
        require(np.all(np.linalg.norm(seq[np.arange(len(ids)),eos.numpy()],axis=-1)>1e-8),'Zero cached EOS text vector')
    write_json(RUN/'cache_audit.json',{'passed':True,'fingerprint':store.key,'queries_verified':len(query_ids),'source_roles':['reference','target'],
        'text_from_trained_checkpoint':True,'all_used_shards_checksum_finite_shape_verified':True,'multilayer_cache_read':False,'all_EOS_text_norms_verified':True})
    return store

def make_batch(rows,store,training=True):
    ids=rows.query_id.tolist();refids=rows.reference_image_id.tolist()
    batch={'reference_features':{'tokens':torch.from_numpy(store.take('reference',refids)).to(DEVICE),'pooled':torch.from_numpy(store.take('reference_pooled',refids)).to(DEVICE)},
        'text_features':torch.from_numpy(store.take('text',ids)).to(DEVICE),'mask':torch.from_numpy(store.take('mask',ids)).to(DEVICE),
        'eos':torch.from_numpy(store.take('eos',ids)).to(DEVICE),'query_ids':ids}
    if training:batch.update(target=torch.from_numpy(store.take('target_pooled',rows.target_image_id.tolist())).to(DEVICE),target_ids=rows.target_image_id.tolist())
    return batch


In [ ]:
# evaluation.py — code nhúng
@torch.inference_mode()
def evaluate_adapter(model,source,store,baseline=False):
    model.eval();rows=[];per={};queries=source['validation'];gallery=source['gallery']
    for category in CATEGORIES:
        frame=queries[queries.category==category];g=gallery[gallery.category==category].sort_values('gallery_index',kind='stable')
        require(g.gallery_index.tolist()==list(range(len(g))) and g.image_id.is_unique,'Gallery alignment')
        lookup={v:i for i,v in enumerate(g.image_id)};vectors=store.take('target_pooled',g.image_id.tolist());ranks=[]
        for start in range(0,len(frame),CFG['eval_batch_size']):
            f=frame.iloc[start:start+CFG['eval_batch_size']];b=make_batch(f,store,training=False)
            require('target' not in b,'Target leaked into inference')
            q=model(b['reference_features'],b['text_features'],b['mask'],b['eos'],gates_off=baseline).cpu().numpy()
            r=exact_ranks(q,vectors,[lookup[v] for v in f.target_image_id]);ranks.extend(r.tolist())
            rows.extend({'query_id':v.query_id,'category':category,'reference_image_id':v.reference_image_id,'target_image_id':v.target_image_id,'target_rank':int(rank)} for v,rank in zip(f.itertuples(),r))
        per[category]={f'R@{k}':float((np.asarray(ranks)<=k).mean()*100) for k in (1,5,10,50)}
    require(len(rows)==len(queries) and {r['query_id'] for r in rows}==set(queries.query_id),'Missing validation queries')
    return {'scope':source.get('scope','fixed full development validation only'),'n_query':len(rows),'per_category':per,**macro_metrics(per),
       'query_counts':queries.groupby('category').size().to_dict(),'gallery_counts':gallery.groupby('category').size().to_dict(),'unit':'percent'},pd.DataFrame(rows)

def verify_baseline(source,store,frozen):
    model=adapter_initial(frozen.projection.weight);start=time.perf_counter();metric,pred=evaluate_adapter(model,source,store,baseline=True)
    expected=EXPECTED_METRICS;deviations={'Mean':metric['Mean']-expected['Mean']}
    for c in CATEGORIES:
        for k in (1,5,10,50):deviations[c+f'/R@{k}']=metric['per_category'][c][f'R@{k}']-expected['per_category'][c][f'R@{k}']
    old=pd.read_csv(source['dev']/'conventional'/'epoch_03_per_query.csv',dtype={'query_id':str})
    merged=pred.merge(old,on=['query_id','category','reference_image_id','target_image_id'],suffixes=('_new','_saved'),validate='one_to_one')
    require(len(merged)==len(pred)==1796,'Baseline prediction mapping mismatch')
    write_csv(RUN/'baseline'/'per_query.csv',pred);write_csv(RUN/'baseline'/'rank_comparison.csv',merged)
    passed=max(abs(v) for v in deviations.values())<=CFG['baseline_max_metric_deviation_pp']
    report={'passed':passed,'metrics':metric,'saved_metrics':expected,'deviation_pp':deviations,'tolerance_pp':CFG['baseline_max_metric_deviation_pp'],
      'rank_difference_count':int((merged.target_rank_new!=merged.target_rank_saved).sum()),'elapsed_seconds':time.perf_counter()-start,'trainable_parameters':0,
      'checkpoint_sha256':EXPECTED_BASE_MARKER['sha256'],'diagnosis_if_failed':'STOP: check strict checkpoint/epoch, captions, reference CLS->postLN/projection, trained text EOS/projection, ID mapping and stable exact ranking. Do not increase tolerance or switch checkpoint.'}
    write_json(RUN/'baseline'/'verification.json',report)
    require(passed,'Baseline reproduction blocked; inspect baseline/verification.json and rank_comparison.csv before training')
    return report


In [ ]:
# training.py — code nhúng
def ensure_fresh_adapter(projection):
    model=adapter_initial(projection);count=sum(p.numel() for p in model.parameters() if p.requires_grad)
    require(count==TRIAL_EXPECTED_PARAMETERS==446978,'Parameter count differs from trial; stop and review architecture')
    reference=read_json(RUN/'verified_trial_reference.json')
    initial=tensor_hash(model)
    if CFG['adapter_seed']==TRIAL_SOURCE_CONFIG['seed']:require(initial==reference['initial_sha256'],'Fresh seed initializer differs from trial; inspect runtime/architecture, do not warm-start')
    return model,initial,count

def training_binding(store,source,initial):
    return {'config':CFG,'code_sha256':CODE_SHA256,'objective':'symmetric multi-positive InfoNCE only','base_checkpoint':EXPECTED_BASE_MARKER['sha256'],
        'base_seed':CFG['base_seed'],'adapter_seed':CFG['adapter_seed'],'initial_sha256':initial,'cache_fingerprint':store.key,
        'train_sha256':source['train_sha256'],'validation_sha256':source['validation_sha256'],'gallery_sha256':source['gallery_sha256']}

def checkpoint_ledger():
    path=RUN/'checkpoint_timing.json';return read_json(path) if path.exists() else []

@contextlib.contextmanager
def epoch_wall_timer(epoch):
    started=time.perf_counter();outcome='completed'
    try:yield
    except BaseException:outcome='failed';raise
    finally:
        path=RUN/'epoch_wall_sessions.json';rows=read_json(path) if path.exists() else []
        rows.append({'epoch':epoch,'elapsed_seconds':time.perf_counter()-started,'status':outcome})
        write_json(path,rows)

def validate_development_resume(state,binding):
    required={'signature','phase','config','binding','epoch','cursor','step','finished','stop_reason','history','metrics','best_mean','best_epoch',
        'bad_evaluations','scheduler','scaler','train_compute_seconds','training_wall_seconds','validation_seconds','model','optimizer','rng'}
    require(required<=state.keys(),'Incomplete development checkpoint: '+', '.join(sorted(required-state.keys())))
    require(state['binding']==binding and state['config']==CFG and state['signature']==fingerprint(binding),'Resume fingerprints/config differ')
    require(state['phase']=='retrieval_only_development' and len(state['history'])==state['step'],'Checkpoint history/phase differs')
    require(state['scheduler'] is None and state['scaler'] is None,'Unexpected scheduler/scaler state')
    require(isinstance(state['optimizer'],dict) and {'state','param_groups'}<=state['optimizer'].keys(),'Optimizer state missing/incompatible')
    require(isinstance(state['rng'],dict) and {'python','numpy','torch','cuda'}<=state['rng'].keys(),'RNG state missing/incompatible')

def _train_development_impl(source,store,frozen,base=None,hashes=None):
    require(read_json(RUN/'baseline'/'verification.json')['passed'],'Baseline reproduction failed')
    tests=read_json(RUN/'test_report.json');require(not tests['failed'] and tests['locked_batch']==CFG['batch_size'],'Preflight failed')
    lock=read_json(RUN/'locked_training_config.json')
    require(lock=={'config':CFG,'code_sha256':CODE_SHA256,'cache_fingerprint':store.key},'Configuration changed after preflight')
    model,initial,count=ensure_fresh_adapter(frozen.projection.weight);binding=training_binding(store,source,initial);signature=fingerprint(binding)
    write_json(RUN/'training_config.json',{'signature':signature,'binding':binding,'trainable_parameters':count,'scheduler':None,'scaler':None,
       'selection':'Dev macro Mean R10/R50, strict improvement; earliest trained epoch on tie; baseline not eligible',
       'recovery_steps_at_risk':CFG['checkpoint_every_steps']-1,'timing_scope':'Compute excludes batch I/O; train wall includes batch fetch/transfer/logging, excludes validation/checkpoint; active pipeline ledger includes all timed phases.'},immutable=True)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG['learning_rate'],weight_decay=CFG['weight_decay'],betas=(.9,.999),eps=1e-8)
    local=LOCAL/'checkpoints';local.mkdir(parents=True,exist_ok=True)
    state={'signature':signature,'phase':'retrieval_only_development','config':CFG,'binding':binding,'epoch':0,'cursor':0,'step':0,'finished':False,
       'stop_reason':None,'history':[],'metrics':[],'best_mean':None,'best_epoch':None,'bad_evaluations':0,'scheduler':None,'scaler':None,
       'train_compute_seconds':0.,'training_wall_seconds':0.,'validation_seconds':0.}
    candidates=[]
    for path in (local/'last.pt',RUN/'last.pt'):
        if path.exists():
            item=checkpoint_load(path,signature);validate_development_resume(item,binding);candidates.append(item)
    if candidates:
        state=max(candidates,key=lambda s:s['step']);model.load_state_dict(state['model'],strict=True);opt.load_state_dict(state['optimizer']);restore_rng(state['rng'])
        require(len(state['history'])==state['step'] and state['phase']=='retrieval_only_development','Checkpoint history/phase differs')
        print('Strict resume epoch',state['epoch'],'cursor',state['cursor'],'step',state['step'])
    else:seed_all(CFG['adapter_seed'])
    del candidates
    def persist_tables():
        records=checkpoint_ledger();checkpoint_seconds=sum(r['elapsed_seconds'] for r in records)
        epoch_path=RUN/'epoch_wall_sessions.json';epoch_records=read_json(epoch_path) if epoch_path.exists() else []
        rows=[]
        for item in state['metrics']:
            epoch=item['epoch'];steps=[r for r in state['history'] if r['epoch']==epoch]
            ck=sum(r['elapsed_seconds'] for r in records if r['epoch']==epoch)
            rows.append({**item,'train_compute_seconds':sum(r['compute_seconds'] for r in steps),'training_wall_seconds':sum(r['training_wall_seconds'] for r in steps),
                'checkpoint_seconds':ck,'epoch_wall_seconds':sum(r['elapsed_seconds'] for r in epoch_records if r['epoch']==epoch) or None,'epoch_active_measured_seconds':sum(r['training_wall_seconds'] for r in steps)+item['validation_seconds']+ck})
        write_csv(RUN/'training_history.csv',pd.DataFrame(state['history']));write_json(RUN/'metric_history.json',rows)
        write_json(RUN/'status.json',{'finished':state['finished'],'stop_reason':state['stop_reason'],'epoch':state['epoch'],'step':state['step'],
           'best_epoch':state['best_epoch'],'best_mean':state['best_mean'],'bad_evaluations':state['bad_evaluations'],'trainable_parameters':count,
           'initial_sha256':initial,'train_compute_seconds':state['train_compute_seconds'],'training_wall_seconds':state['training_wall_seconds'],
           'validation_seconds':state['validation_seconds'],'checkpoint_seconds':checkpoint_seconds,'peak_allocated_bytes':max([i['peak_allocated_bytes'] or 0 for i in rows],default=0) or None,
           'timing_note':'Observed active durations only; unrecorded time from abruptly lost runtimes is unavailable, not inferred.'})
    def commit(best=False,epoch_label=None):
        started=time.perf_counter();outcome='completed'
        try:
            state.update(model=model.state_dict(),optimizer=opt.state_dict(),rng=rng_state())
            checkpoint_save(local/'last.pt',state,signature);sync_checkpoint(local/'last.pt',RUN/'last.pt',signature)
            if best:sync_checkpoint(local/'last.pt',RUN/'best.pt',signature)
        except BaseException:outcome='failed';raise
        finally:
            records=checkpoint_ledger();records.append({'epoch':epoch_label if epoch_label is not None else state['epoch']+1,'step':state['step'],
                'elapsed_seconds':time.perf_counter()-started,'best_written':best,'status':outcome})
            write_json(LOCAL/'checkpoint_timing.json',records);write_json(RUN/'checkpoint_timing.json',records)
        persist_tables()
    indexed=source['train'].set_index('query_id',drop=False)
    try:
        for epoch in range(state['epoch'],CFG['max_epochs']):
            if state['finished']:break
            with epoch_wall_timer(epoch+1):
                batches=epoch_batches(source['train'],CFG['batch_size'],CFG['adapter_seed'],epoch)
                require(state['cursor']<=len(batches),'Invalid checkpoint cursor')
                if DEVICE.type=='cuda':torch.cuda.reset_peak_memory_stats()
                for cursor in range(state['cursor'],len(batches)):
                    wall_started=time.perf_counter();rows=indexed.loc[batches[cursor]]
                    require(rows.query_id.tolist()==batches[cursor] and set(rows.official_split)=={'train'},'Batch identity/split differs')
                    batch=make_batch(rows,store);model.train();opt.zero_grad(set_to_none=True)
                    if DEVICE.type=='cuda':torch.cuda.synchronize()
                    started=time.perf_counter();loss=objective(model,batch);loss.backward();norms=gradient_summary(model)
                    grad=torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['gradient_clip'],error_if_nonfinite=True);opt.step()
                    if DEVICE.type=='cuda':torch.cuda.synchronize()
                    compute=time.perf_counter()-started
                    values=torch.stack([loss.detach(),grad.detach(),torch.tanh(model.gate_ri).detach(),torch.tanh(model.gate_ta).detach()]).cpu().tolist()
                    wall=time.perf_counter()-wall_started;state['train_compute_seconds']+=compute;state['training_wall_seconds']+=wall;state['step']+=1;state['cursor']=cursor+1
                    state['history'].append({'epoch':epoch+1,'step':state['step'],'cursor':cursor+1,'batch_ids_sha256':fingerprint(batches[cursor]),
                        'loss_retrieval':values[0],'gradient_norm':values[1],'gate_RI':values[2],'gate_TA':values[3],'lr':opt.param_groups[0]['lr'],
                        'compute_seconds':compute,'training_wall_seconds':wall,**norms})
                    if state['step']%CFG['checkpoint_every_steps']==0:commit(epoch_label=epoch+1)
                model.eval();started=time.perf_counter();metric,pred=evaluate_adapter(model,source,store);evaluation=time.perf_counter()-started;state['validation_seconds']+=evaluation
                require(metric['n_query']==len(source['validation']),'Full validation coverage required')
                improved=state['best_mean'] is None or metric['Mean']>state['best_mean']
                if improved:state.update(best_mean=metric['Mean'],best_epoch=epoch+1,bad_evaluations=0)
                else:state['bad_evaluations']+=1
                state['metrics'].append({'epoch':epoch+1,'metrics':metric,'validation_seconds':evaluation,
                    'peak_allocated_bytes':torch.cuda.max_memory_allocated() if DEVICE.type=='cuda' else None})
                write_csv(RUN/f'epoch_{epoch+1:02d}_per_query.csv',pred)
                done=(epoch+1>=CFG['max_epochs'] or state['bad_evaluations']>=CFG['patience'])
                state.update(epoch=epoch+1,cursor=0,finished=done,stop_reason=('patience' if state['bad_evaluations']>=CFG['patience'] else 'max_epochs') if done else None)
                commit(best=improved,epoch_label=epoch+1)
                if base is not None:assert_backbones_frozen(base,frozen,hashes)
                print('Epoch',epoch+1,'Dev Mean',metric['Mean'],'best',state['best_epoch'],'patience',state['bad_evaluations'],'/',CFG['patience'])
        persist_tables();return {k:v for k,v in state.items() if k not in ('model','optimizer','rng')}
    except BaseException:
        write_bytes(RUN/'training_traceback.txt',traceback.format_exc().encode());raise
    finally:opt.zero_grad(set_to_none=True)


def train_development(source,store,frozen,base=None,hashes=None):
    require_ready_for_training(source,store,frozen,base,hashes)
    return _train_development_impl(source,store,frozen,base,hashes)


In [ ]:
# exports.py — code nhúng
def load_best_development(frozen,store,source):
    config=read_json(RUN/'training_config.json');binding=config['binding']
    require(config['signature']==fingerprint(binding) and binding['config']==CFG and binding['code_sha256']==CODE_SHA256,'Best config mismatch')
    require(binding['base_checkpoint']==EXPECTED_BASE_MARKER['sha256'] and binding['cache_fingerprint']==store.key and binding['train_sha256']==source['train_sha256'],'Best provenance mismatch')
    state=checkpoint_load(RUN/'best.pt',config['signature']);require(state['binding']==binding and state['best_epoch']>=1,'Best must be a trained epoch, never baseline')
    model,initial,count=ensure_fresh_adapter(frozen.projection.weight);require(initial==binding['initial_sha256'],'Best initializer provenance')
    model.load_state_dict(state['model'],strict=True);model.eval();return model,state

@torch.inference_mode()
def evaluate_topk(model,source,store):
    model.eval();rows=[];top=[];per={}
    for category in CATEGORIES:
        queries=source['validation'][source['validation'].category==category];g=source['gallery'][source['gallery'].category==category].sort_values('gallery_index',kind='stable')
        require(g.gallery_index.tolist()==list(range(len(g))) and g.image_id.is_unique,'TopK gallery ID/order')
        ids=g.image_id.tolist();lookup={v:i for i,v in enumerate(ids)};vectors=store.take('target_pooled',ids);vectors=vectors/np.linalg.norm(vectors,axis=-1,keepdims=True);ranks=[]
        for start in range(0,len(queries),CFG['eval_batch_size']):
            f=queries.iloc[start:start+CFG['eval_batch_size']];batch=make_batch(f,store,training=False);require('target' not in batch,'Target entered query')
            q=model.encode_query(batch['reference_features'],batch['text_features'],batch['mask'],batch['eos']).cpu().numpy();q=q/np.linalg.norm(q,axis=-1,keepdims=True)
            scores=q@vectors.T;require(np.isfinite(scores).all(),'Nonfinite ranking scores')
            positions=[lookup[v] for v in f.target_image_id];targets=scores[np.arange(len(f)),positions]
            rank=1+(scores>targets[:,None]).sum(1)+((scores==targets[:,None])&(np.arange(len(ids))[None,:]<np.array(positions)[:,None])).sum(1);ranks.extend(rank.tolist())
            order=np.argsort(-scores,axis=1,kind='stable')[:,:min(CFG['export_top_k'],len(ids))]
            for i,row in enumerate(f.itertuples()):
                rows.append({'query_id':row.query_id,'category':category,'reference_image_id':row.reference_image_id,'target_image_id':row.target_image_id,'target_rank':int(rank[i])})
                top.extend({'query_id':row.query_id,'category':category,'rank':j+1,'image_id':ids[position],'score':float(scores[i,position])} for j,position in enumerate(order[i]))
        per[category]={f'R@{k}':float((np.asarray(ranks)<=k).mean()*100) for k in (1,5,10,50)}
    require(len(rows)==len(source['validation']) and len({r['query_id'] for r in rows})==len(rows),'TopK query coverage')
    return {'n_query':len(rows),'unit':'percent','per_category':per,**macro_metrics(per)},pd.DataFrame(rows),pd.DataFrame(top)

def export_best_results(source,store,frozen):
    model,state=load_best_development(frozen,store,source);metric,queries,top=evaluate_topk(model,source,store)
    selected=next(x for x in state['metrics'] if x['epoch']==state['best_epoch'])['metrics']
    require(abs(metric['Mean']-selected['Mean'])<1e-10 and all(abs(metric['per_category'][c][k]-selected['per_category'][c][k])<1e-10 for c in CATEGORIES for k in metric['per_category'][c]),'Best checkpoint reranking differs')
    write_csv(RUN/'per_query_results.csv',queries);write_csv(RUN/'top_k_results.csv',top)
    category_rows=[{'category':c,**m,'Mean':(m['R@10']+m['R@50'])/2} for c,m in metric['per_category'].items()]
    category_rows.append({'category':'Overall macro',**metric['overall'],'Mean':metric['Mean']})
    write_csv(RUN/'per_category_metrics.csv',pd.DataFrame(category_rows))
    # Read back persisted result artifacts and verify metric/ID/TopK alignment.
    saved=pd.read_csv(RUN/'per_query_results.csv',dtype={'query_id':str});saved_top=pd.read_csv(RUN/'top_k_results.csv',dtype={'query_id':str,'image_id':str})
    require(saved.query_id.tolist()==queries.query_id.tolist() and len(saved_top)==len(top),'Result readback coverage differs')
    for c in CATEGORIES:
        r=saved[saved.category==c].target_rank.to_numpy()
        require(all(abs(float((r<=k).mean()*100)-metric['per_category'][c][f'R@{k}'])<1e-10 for k in (1,5,10,50)),'Saved metric readback mismatch')
        require(set(saved_top[saved_top.category==c].image_id)<=set(source['gallery'][source['gallery'].category==c].image_id),'TopK wrong category gallery')
    counts=saved_top.groupby('query_id').size();require(len(counts)==len(saved) and counts.min()==min(CFG['export_top_k'],len(source['gallery'][source['gallery'].category==CATEGORIES[0]])),'TopK length mismatch')
    outputs={name:file_hash(RUN/name) for name in ('per_query_results.csv','top_k_results.csv','per_category_metrics.csv')}
    report={'passed':True,'best_epoch':state['best_epoch'],'metrics':metric,'checkpoint_sha256':read_json(RUN/'best.json')['sha256'],'outputs':outputs,'query_coverage':len(saved),'top_k':CFG['export_top_k']}
    write_json(RUN/'best_result_verification.json',report);del model,state;return report

def completion_audit():
    failures=[]
    try:
        status=read_json(RUN/'status.json');result=read_json(RUN/'best_result_verification.json');config=read_json(RUN/'training_config.json')
        require(status.get('finished') is True,'Training status not finished')
        require(result.get('passed') is True and result['query_coverage']==1796 and result['metrics']['n_query']==1796,'Best result incomplete')
        require(config['signature']==fingerprint(config['binding']) and config['binding']['config']==CFG and config['binding']['code_sha256']==CODE_SHA256,'Completion config mismatch')
        require(status['best_epoch']==result['best_epoch'] and status['best_epoch']>=1,'Best epoch mismatch')
        for name in ('best','last'):
            marker=read_json(RUN/(name+'.json'))
            require(marker['signature']==config['signature'] and file_hash(RUN/(name+'.pt'))==marker['sha256'],'Completion checkpoint invalid: '+name)
        require(result['checkpoint_sha256']==read_json(RUN/'best.json')['sha256'],'Verified result belongs to another checkpoint')
        for name,sha in result['outputs'].items():
            require(Path(name).name==name and file_hash(RUN/name)==sha,'Output missing/corrupt: '+name)
        required={'per_query_results.csv','top_k_results.csv','per_category_metrics.csv'}
        require(required<=set(result['outputs']),'Result verification omits required artifacts')
        q=pd.read_csv(RUN/'per_query_results.csv',dtype={'query_id':str});manifest=pd.read_csv(RUN/'manifests'/'validation.csv',dtype=str,keep_default_na=False)
        require(len(q)==len(manifest)==1796 and q.query_id.is_unique and set(q.query_id)==set(manifest.query_id),'Completion query coverage mismatch')
        require(q[['query_id','category','reference_image_id','target_image_id']].merge(manifest,on=['query_id','category','reference_image_id','target_image_id'],validate='one_to_one').shape[0]==1796,'Completion mapping mismatch')
        for c in CATEGORIES:
            ranks=q[q.category==c].target_rank.to_numpy()
            require(len(ranks)==EXPECTED_METRICS['query_counts'][c] and np.isfinite(ranks).all() and np.all(ranks==ranks.astype(int)) and np.all((ranks>=1)&(ranks<=EXPECTED_METRICS['gallery_counts'][c])),'Completion category coverage/rank invalid')
            require(all(abs(float((ranks<=k).mean()*100)-result['metrics']['per_category'][c][f'R@{k}'])<1e-10 for k in (1,5,10,50)),'Completion saved metric mismatch')
        top=pd.read_csv(RUN/'top_k_results.csv',dtype={'query_id':str,'image_id':str})
        galleries=pd.read_csv(RUN/'manifests'/'gallery.csv',dtype={'image_id':str})
        require(set(top.query_id)==set(q.query_id) and not top.duplicated(['query_id','rank']).any() and not top.duplicated(['query_id','image_id']).any(),'Completion TopK coverage/duplicates')
        require(len(top.merge(q[['query_id','category']],on=['query_id','category'],validate='many_to_one'))==len(top),'Completion TopK category mismatch')
        for c in CATEGORIES:
            part=top[top.category==c];k=min(CFG['export_top_k'],EXPECTED_METRICS['gallery_counts'][c])
            require(set(part.image_id)<=set(galleries[galleries.category==c].image_id) and np.isfinite(part.score).all(),'Completion TopK image/score invalid')
            require(len(part)==EXPECTED_METRICS['query_counts'][c]*k and part.groupby('query_id')['rank'].apply(lambda r:sorted(r)==list(range(1,k+1))).all(),'Completion TopK rank/count mismatch')
        category_csv=pd.read_csv(RUN/'per_category_metrics.csv').set_index('category')
        require(set(category_csv.index)==set(CATEGORIES)|{'Overall macro'} and category_csv.index.is_unique,'Completion category CSV schema')
        for c in (*CATEGORIES,'Overall macro'):
            expected=result['metrics']['per_category'][c] if c in CATEGORIES else result['metrics']['overall']
            require(all(abs(category_csv.loc[c,f'R@{k}']-expected[f'R@{k}'])<1e-10 for k in (1,5,10,50)) and abs(category_csv.loc[c,'Mean']-(expected['R@10']+expected['R@50'])/2)<1e-10,'Completion category CSV metric mismatch')
        aggregate=macro_metrics(result['metrics']['per_category'])
        require(all(abs(aggregate['overall'][f'R@{k}']-result['metrics']['overall'][f'R@{k}'])<1e-10 for k in (1,5,10,50)) and abs(aggregate['Mean']-result['metrics']['Mean'])<1e-10,'Completion macro/Mean mismatch')
        require(abs(status['best_mean']-result['metrics']['Mean'])<1e-10,'Completion selected score mismatch')
    except Exception as exc:failures.append(str(exc))
    return {'verified_complete':not failures,'failures':failures}

def development_summary():
    base=read_json(RUN/'baseline'/'verification.json');trial=read_json(RUN/'verified_trial_reference.json');rows=[]
    def row(label,m,n,epoch,compute,wall,validation,checkpoint):
        return {'configuration':label,'base_seed':CFG['base_seed'],'adapter_seed':None if n==0 else trial['seed'] if n==2048 else CFG['adapter_seed'],
          'adapter_budget_epochs':0 if n==0 else trial['budget_epochs'] if n==2048 else CFG['max_epochs'],
          'trainable_parameters':0 if n==0 else TRIAL_EXPECTED_PARAMETERS,'adapter_train_samples':n,'encoder_train_samples':16204,'selected_epoch':epoch,
          'selected_epoch_component':'encoder' if n==0 else 'adapter',
          'R@10':m['overall']['R@10'],'R@50':m['overall']['R@50'],'Mean':m['Mean'],'delta_vs_frozen':m['Mean']-base['metrics']['Mean'],
          'train_compute_seconds':compute,'training_wall_seconds':wall,'validation_seconds':validation,'checkpoint_seconds':checkpoint}
    rows.append(row('Frozen Conventional-only',base['metrics'],0,3,0.,0.,base['elapsed_seconds'],0.))
    rows.append(row('Retrieval-only trial (2048, budget3)',trial['metrics'],2048,trial['best_epoch'],trial['train_compute_seconds'],None,trial['validation_seconds'],None))
    status=read_json(RUN/'status.json') if (RUN/'status.json').exists() else None
    result=read_json(RUN/'best_result_verification.json') if (RUN/'best_result_verification.json').exists() else None
    completion=completion_audit()
    if result and completion['verified_complete']:
        rows.append(row(f'Retrieval-only development (16204, budget{CFG["max_epochs"]})',result['metrics'],16204,result['best_epoch'],status['train_compute_seconds'],status['training_wall_seconds'],status['validation_seconds'],status['checkpoint_seconds']))
    else:rows.append({'configuration':f'Retrieval-only development (16204, budget{CFG["max_epochs"]})','base_seed':CFG['base_seed'],'adapter_seed':CFG['adapter_seed'],'adapter_budget_epochs':CFG['max_epochs'],'trainable_parameters':TRIAL_EXPECTED_PARAMETERS,'selected_epoch_component':'adapter','adapter_train_samples':16204,'encoder_train_samples':16204,'selected_epoch':None,'R@10':None,'R@50':None,'Mean':None,'delta_vs_frozen':None,'train_compute_seconds':None,'training_wall_seconds':None,'validation_seconds':None,'checkpoint_seconds':None})
    phases=read_json(RUN/'pipeline_timing_sessions.json') if (RUN/'pipeline_timing_sessions.json').exists() else []
    timings={'phases':phases,'active_pipeline_wall_seconds':sum(r['elapsed_seconds'] for r in phases),
      'cache_wall_seconds':sum(r['elapsed_seconds'] for r in phases if r['phase']=='cache'),
      'definition':'Non-overlapping active phase wall durations include cache/source validation, encoder loads, baseline/trial validation, preflight, training I/O/compute, validation/checkpoint, best export. Excludes notebook idle; abruptly unrecorded lost-runtime time is unavailable.',
      'training':status,'epochs':read_json(RUN/'metric_history.json') if (RUN/'metric_history.json').exists() else []}
    write_json(RUN/'timing_resource_report.json',timings)
    summary={'results':rows,'finished':completion['verified_complete'],'completion_audit':completion,'base_seed':CFG['base_seed'],'adapter_seed':CFG['adapter_seed'],
       'train_count':16204,'val_count':1796,'gallery_counts':{c:EXPECTED_METRICS['gallery_counts'][c] for c in CATEGORIES},
       'best_trained_adapter_below_baseline':bool(completion['verified_complete'] and result and result['metrics']['Mean']<base['metrics']['Mean']),
       'same_validation_gallery':trial['same_validation_gallery'],'budgets_differ':True,'official_val_used':False,'heads_trained':False,
       'seed_scope':'Changing adapter seeds with this fixed base encoder measures adapter variation only; not multi-seed full-pipeline evidence.',
       'historical_timing_note':trial['historical_timing_note'],'timing':timings}
    write_csv(RUN/'comparison.csv',pd.DataFrame(rows));write_json(RUN/'development_summary.json',summary)
    text='# Adapter retrieval-only development\n\n'
    text+=f"Base seed {CFG['base_seed']}; adapter seed {CFG['adapter_seed']}. Train16204 / val1796; budget {CFG['max_epochs']} epoch, patience {CFG['patience']}. Finished: {summary['finished']}.\n\n"
    for r in rows:text+=f"- {r['configuration']}: best epoch {r['selected_epoch']}, Mean {r['Mean']}, delta {r['delta_vs_frozen']}.\n"
    text+='\nHistorical trial uses2048/3epoch; development uses16204/max10. Same validation/gallery, different budgets. Negative deltas retained; baseline is never an eligible trained-adapter checkpoint. See timing_resource_report.json; historical compute is not total wall time. One fixed base seed; no statistical or full-pipeline multi-seed claim.\n'
    write_bytes(RUN/'summary.md',text.encode());print(pd.DataFrame(rows).round(5).to_string(index=False));return summary

@torch.inference_mode()
def export_enhanced(model,frame,store,destination,source):
    require(frame.query_id.is_unique and set(frame.official_split)=={'train'} and set(frame.query_id)<=set(source['train'].query_id)|set(source['validation'].query_id),'Export outside development manifests')
    binding={'model_sha256':tensor_hash(model),'cache':store.key,'base_checkpoint':EXPECTED_BASE_MARKER['sha256'],'query_ids':frame.query_id.tolist(),'features':'enhanced text sequence, not composed vector'}
    key=fingerprint(binding);root=Path(destination)/key;root.mkdir(parents=True,exist_ok=True);write_json(root/'index.json',binding,immutable=True);model.eval()
    for start in range(0,len(frame),128):
        rows=frame.iloc[start:start+128];p=root/f'{start:07d}.npz';marker=p.with_suffix('.json')
        if p.exists():require(marker.exists() and read_json(marker)['fingerprint']==key and file_hash(p)==read_json(marker)['sha256'],'Existing export corrupt');continue
        values={k:[] for k in ('sequence','mask','eos')}
        for offset in range(0,len(rows),CFG['eval_batch_size']):
            b=make_batch(rows.iloc[offset:offset+CFG['eval_batch_size']],store,training=False);h,m,e=model.enhanced_text_sequence(b['reference_features'],b['text_features'],b['mask'],b['eos'])
            for name,v in [('sequence',h),('mask',m),('eos',e)]:values[name].append(v.cpu().numpy())
        temp=p.with_suffix('.partial.npz');np.savez(temp,**{k:np.concatenate(v) for k,v in values.items()},query_id=rows.query_id.to_numpy(dtype=str))
        os.replace(temp,p);write_json(marker,{'sha256':file_hash(p),'fingerprint':key,'query_ids':rows.query_id.tolist()})
    return root


In [ ]:
# tests_runtime.py — code nhúng
def query_from_batch(model,batch,**kw):return model(batch['reference_features'],batch['text_features'],batch['mask'],batch['eos'],**kw)

def adapter_smoke(projection,batch):
    saved=rng_state();model=adapter_initial(projection);model.eval();initial=tensor_hash(model)
    try:
        with torch.no_grad():
            idx=torch.arange(len(batch['eos']),device=DEVICE);baseline=unit(unit(F.linear(batch['text_features'][idx,batch['eos']],projection))+unit(batch['reference_features']['pooled']))
            require(torch.allclose(query_from_batch(model,batch),baseline,atol=CFG['gate_equivalence_atol'],rtol=0),'Gate-off baseline changed')
            one={k:({p:v[:1] for p,v in value.items()} if isinstance(value,dict) else value[:1]) for k,value in batch.items()}
            require(query_from_batch(model,one).shape==(1,projection.shape[0]),'Batch1 shape')
            changed={**batch,'target':batch['target'].flip(0)};require(torch.equal(query_from_batch(model,batch),query_from_batch(model,changed)),'Target affects query')
            dirty={**batch,'text_features':batch['text_features'].clone()};dirty['text_features'][batch['mask']==0]=12345.
            require(torch.equal(query_from_batch(model,batch),query_from_batch(model,dirty)),'Padding changes pooling')
        opt=torch.optim.AdamW(model.parameters(),lr=CFG['learning_rate'],weight_decay=CFG['weight_decay']);norms=[]
        for step in range(4):
            opt.zero_grad(set_to_none=True);teacher=batch['target'].detach().clone().requires_grad_();loss=objective(model,{**batch,'target':teacher});loss.backward()
            require(teacher.grad is None and all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in model.parameters()),'Missing/nonfinite grad or teacher attached')
            norm=gradient_summary(model);norms.append(norm)
            if step==0:require(norm['RI']==0 and norm['TA']==0,'Zero-gated branch unexpectedly active at first step')
            torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['gradient_clip'],error_if_nonfinite=True);opt.step()
        require(norms[-1]['RI']>0 and norms[-1]['TA']>0,'RI/TA disconnected after gates update')
        with torch.no_grad():require(torch.allclose(query_from_batch(model,batch,gates_off=True),baseline,atol=CFG['gate_equivalence_atol'],rtol=0),'Forced gate-off changed base')
        with tempfile.TemporaryDirectory() as d:
            path=Path(d)/'smoke.pt';sig=fingerprint({'fixture':'retrieval-adapter-resume'})
            checkpoint_save(path,{'signature':sig,'phase':'smoke','step':4,'model':model.state_dict(),'optimizer':opt.state_dict(),'rng':rng_state()},sig)
            def step_once():
                opt.zero_grad(set_to_none=True);scale=1+.01*(torch.rand((),device=DEVICE)+random.random()+float(np.random.random()))
                loss=objective(model,batch)*scale;loss.backward();opt.step()
            step_once();expected={k:v.detach().clone() for k,v in model.state_dict().items()};er=rng_state()
            state=checkpoint_load(path,sig);model.load_state_dict(state['model'],strict=True);opt.load_state_dict(state['optimizer']);restore_rng(state['rng']);step_once();ar=rng_state()
            require(all(torch.equal(v,model.state_dict()[k]) for k,v in expected.items()),'Resume model/optimizer differs')
            require(er['python']==ar['python'] and np.array_equal(er['numpy'][1],ar['numpy'][1]) and er['numpy'][2:]==ar['numpy'][2:] and torch.equal(er['torch'],ar['torch']),'CPU RNG restore differs')
            if er['cuda'] is not None:require(all(torch.equal(a,b) for a,b in zip(er['cuda'],ar['cuda'])),'CUDA RNG restore differs')
        return {'passed':True,'gate_off_equivalence':True,'B1_and_normal':True,'target_independence':True,'teacher_detached':True,'padding_pooling_invariant':True,
          'gate_gradient_stages':norms,'optimizer_rng_save_load':True,'initial_sha256':initial,'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad)}
    finally:restore_rng(saved)

def preflight_development(source,store,frozen,base,hashes):
    saved=rng_state();report={'passed':[],'failed':[],'skipped':[]}
    try:
        require(read_json(RUN/'baseline'/'verification.json')['passed'],'Baseline reproduction required')
        report['passed'].extend([pure_tests(),storage_tests()])
        rows=stratified_sample(source['train'],CFG['batch_size'],CFG['adapter_seed']);batch=make_batch(rows,store)
        smoke=adapter_smoke(frozen.projection.weight,batch);require(smoke['trainable_parameters']==TRIAL_EXPECTED_PARAMETERS,'Parameter count changed')
        report['passed'].append(smoke)
        model,initial,count=ensure_fresh_adapter(frozen.projection.weight);opt=torch.optim.AdamW(model.parameters(),lr=CFG['learning_rate'],weight_decay=CFG['weight_decay'])
        if DEVICE.type=='cuda':torch.cuda.reset_peak_memory_stats()
        for _ in range(2):
            opt.zero_grad(set_to_none=True);objective(model,batch).backward();torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['gradient_clip'],error_if_nonfinite=True);opt.step()
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        started=time.perf_counter()
        for _ in range(3):
            opt.zero_grad(set_to_none=True);objective(model,batch).backward();torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['gradient_clip'],error_if_nonfinite=True);opt.step()
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        elapsed=time.perf_counter()-started
        report['profile']={'batch_size':CFG['batch_size'],'three_steps_seconds':elapsed,'examples_per_second':3*CFG['batch_size']/elapsed,
           'estimated_epoch_compute_seconds':elapsed/3*math.ceil(len(source['train'])/CFG['batch_size']),
           'peak_allocated_bytes':torch.cuda.max_memory_allocated() if DEVICE.type=='cuda' else None,'scope':'Temporary adapter compute; excludes full batch I/O, validation, checkpoint and pipeline; not measured epoch wall time.'}
        assert_backbones_frozen(base,frozen,hashes);report['passed'].append({'frozen_backbone_hashes_unchanged':True,'temporary_RNG_initialization':True})
        report['locked_batch']=CFG['batch_size'];report['initial_sha256']=initial;report['base_seed']=CFG['base_seed'];report['adapter_seed']=CFG['adapter_seed']
        write_json(RUN/'locked_training_config.json',{'config':CFG,'code_sha256':CODE_SHA256,'cache_fingerprint':store.key},immutable=True)
    except BaseException:report['failed'].append(traceback.format_exc());raise
    finally:restore_rng(saved);write_json(RUN/'test_report.json',report)
    print('Preflight passed, adapter params:',count,'| base seed',CFG['base_seed'],'adapter seed',CFG['adapter_seed'])
    return report


def storage_tests():
    saved=rng_state()
    try:
        with tempfile.TemporaryDirectory() as d:
            root=Path(d);spec={'x':{'ids':['a','b'],'shape':[3],'dtype':'float32'}}
            st=TrialStore(root/'remote',root/'local',{'fixture':42},spec,2);st.prepare('x',lambda ids:np.ones((len(ids),3),np.float32))
            require(st.take('x',['b','a']).shape==(2,3),'Cache ID read')
            marker=st.root/'x'/'0000000_0000002.json';require(marker.is_file(),'Committed fixture marker missing');original=marker.read_bytes();m=read_json(marker);m['fingerprint']='wrong';marker.write_text(json.dumps(m))
            try:st.prepare('x',lambda ids:None)
            except RuntimeError:pass
            else:raise AssertionError('Wrong fingerprint accepted')
            marker.write_bytes(original);data=marker.with_suffix('.npy');raw=data.read_bytes();data.write_bytes(raw[:-2]+b'xx')
            try:st.prepare('x',lambda ids:None)
            except RuntimeError:pass
            else:raise AssertionError('Corrupt cache accepted')
            data.write_bytes(raw);pending=marker.with_suffix('.pending.json');pending.write_bytes(original);marker.unlink();st.prepare('x',lambda ids:None)
            require(marker.is_file(),'Pending commit not recovered')
        return {'cache_corruption_blocked':True,'wrong_fingerprint_blocked':True,'cache_pending_commit_recovery':True,'committed_marker_selected_explicitly':True}
    finally:restore_rng(saved)


In [ ]:
# validation.py — kiểm tra tạm và runtime gate
READY_FOR_TRAINING=False
_VALIDATION_RECEIPT=None

def _frame_sha(frame):
    return hashlib.sha256(frame.to_csv(index=False,lineterminator='\n').encode()).hexdigest()

def validation_binding(source,store,hashes):
    return {'session_id':VALIDATION_SESSION_ID,'config':CFG,'code_sha256':CODE_SHA256,'run':str(RUN),
        'cache_fingerprint':store.key,'base_checkpoint':EXPECTED_BASE_MARKER['sha256'],'backbone_hashes':hashes,
        'train_sha256':_frame_sha(source['train']),'validation_sha256':_frame_sha(source['validation']),'gallery_sha256':_frame_sha(source['gallery'])}

def seal_cache(store,source):
    queries=set(source['train'].query_id)|set(source['validation'].query_id)
    require(len(source['train'])==16204 and len(source['validation'])==1796 and len(queries)==18000,'Required full development coverage')
    require(all(set(store.maps[r])==queries for r in ('text','mask','eos','input_ids')),'Text/mask/EOS coverage mismatch')
    require(set(source['train'].reference_image_id)|set(source['validation'].reference_image_id)<=set(store.maps['reference']),'Reference coverage missing')
    require(set(source['train'].target_image_id)|set(source['gallery'].image_id)<=set(store.maps['target_pooled']),'Target/gallery coverage missing')
    require(set(store.maps['reference'])==set(store.maps['reference_pooled']),'Reference pooled ID alignment')
    require(store.contract['base_checkpoint_sha256']==EXPECTED_BASE_MARKER['sha256'],'Cache uses another trained encoder')
    require(store.contract['train_sha256']==source['train_sha256'] and store.contract['validation_sha256']==source['validation_sha256'] and store.contract['gallery_sha256']==source['gallery_sha256'],'Cache manifest fingerprint differs')
    expected={(r,start) for r,s in store.specs.items() for start in range(0,len(s['ids']),store.step)}
    require(set(store.paths)==expected,'Some cache shards have not completed verification')
    audit=read_json(RUN/'cache_audit.json');require(audit['passed'] and audit['fingerprint']==store.key and audit['queries_verified']==18000 and audit.get('all_EOS_text_norms_verified') is True,'No complete cache audit')
    # All array checksums/shapes/finiteness/norms were checked by prepare_cache in this process.
    # A local stat seal detects accidental changes before training without hashing Drive in batches.
    return {str(path):(path.stat().st_size,path.stat().st_mtime_ns) for path in store.paths.values()}

def isolated_training_smoke(source,store,frozen):
    """Real cached subset, temporary adapter/config/RUN and local checkpoint mirror. No main checkpoint writes."""
    global CFG,RUN,LOCAL,sync_checkpoint
    saved_cfg,saved_run,saved_local=CFG,RUN,LOCAL;saved_rng=rng_state();original_sync=sync_checkpoint
    require(read_json(saved_run/'baseline'/'verification.json')['passed'],'Full baseline must pass before smoke')
    train=stratified_sample(source['train'],8,CFG['adapter_seed'])
    val=stratified_sample(source['validation'],6,CFG['adapter_seed'])
    gallery=[]
    for c in CATEGORIES:
        rows=source['gallery'][(source['gallery'].category==c)&source['gallery'].image_id.isin(val[val.category==c].target_image_id)].copy()
        rows=rows.sort_values('gallery_index',kind='stable');rows['gallery_index']=np.arange(len(rows))
        require(1<=len(rows)<=2,'Bounded smoke gallery must cover selected targets');gallery.append(rows)
    g=pd.concat(gallery,ignore_index=True)
    fixture={'train':train,'validation':val,'gallery':g,'train_sha256':_frame_sha(train),'validation_sha256':_frame_sha(val),'gallery_sha256':_frame_sha(g),
        'scope':'pre-training diagnostic only: 8 real cached training rows, 6 validation rows, <=2 gallery images/category; not full-development metrics'}
    receipt=read_json(saved_run/'verified_trial_reference.json')
    try:
        with tempfile.TemporaryDirectory(prefix='pretraining_smoke_',dir=saved_local) as tmp:
            root=Path(tmp);CFG={**saved_cfg,'batch_size':4,'max_epochs':4,'patience':3,'checkpoint_every_steps':2,'eval_batch_size':4,
                'experiment_id':saved_cfg['experiment_id']+'_bounded_validation_fixture'}
            def setup(label):
                global RUN,LOCAL
                RUN=root/label;LOCAL=root/(label+'_local');RUN.mkdir();LOCAL.mkdir()
                write_json(RUN/'verified_trial_reference.json',receipt)
                temp,initial,count=ensure_fresh_adapter(frozen.projection.weight)
                baseline,pred=evaluate_adapter(temp,fixture,store,baseline=True)
                require(baseline['n_query']==len(val) and np.isfinite(pred.target_rank.to_numpy()).all(),'Smoke baseline evaluator failed')
                write_json(RUN/'baseline'/'verification.json',{'passed':True,'metrics':baseline,'scope':fixture['scope']})
                write_json(RUN/'test_report.json',{'failed':[],'locked_batch':4,'scope':fixture['scope'],'passed':['Parent full baseline and real batch preflight already passed; local bounded runner fixture']})
                write_json(RUN/'locked_training_config.json',{'config':CFG,'code_sha256':CODE_SHA256,'cache_fingerprint':store.key})
                del temp
            setup('continuous')
            completed=_train_development_impl(fixture,store,frozen)
            require(completed['finished'] and completed['epoch']==4 and completed['stop_reason']=='patience' and completed['best_epoch']==1 and completed['step']==8,'Real bounded early-stop/tie fixture failed')
            require(all(x['metrics']['Mean']==100. for x in completed['metrics']),'Tiny-gallery fixture must tie at R10/R50=100; never a scientific metric')
            config=read_json(RUN/'training_config.json');state=checkpoint_load(RUN/'last.pt',config['signature'])
            expected={k:v.detach().cpu().clone() for k,v in state['model'].items()};expected_history=state['history'];del state
            before=completed['step'];require(_train_development_impl(fixture,store,frozen)['step']==before,'Resume added optimizer updates')
            result=export_best_results(fixture,store,frozen)
            require(result['passed'] and result['query_coverage']==6,'Smoke CSV/JSON/TopK readback failed')
            setup('interrupted');fault=[True]
            def stop_once(local,dest,signature):
                if fault[0] and read_json(Path(local).with_suffix('.json'))['step']==2:
                    fault[0]=False;raise RuntimeError('controlled_pretraining_smoke_interruption')
                return original_sync(local,dest,signature)
            sync_checkpoint=stop_once
            try:_train_development_impl(fixture,store,frozen)
            except RuntimeError as exc:require(str(exc)=='controlled_pretraining_smoke_interruption','Unexpected smoke failure: '+str(exc))
            else:raise AssertionError('Smoke interruption was not triggered')
            finally:sync_checkpoint=original_sync
            resumed=_train_development_impl(fixture,store,frozen)
            state=checkpoint_load(RUN/'last.pt',read_json(RUN/'training_config.json')['signature'])
            require(resumed['step']==8 and all(torch.equal(v,state['model'][k].detach().cpu()) for k,v in expected.items()),'Mid-epoch optimizer/model restore differs')
            for a,b in zip(expected_history,state['history']):
                require(all(a[k]==b[k] for k in a if k not in ('compute_seconds','training_wall_seconds')),'Resume data order/loss/gradient history differs')
            del state
            return {'passed':True,'scope':fixture['scope'],'training_rows':8,'validation_rows':6,'continuous_optimizer_steps':8,'resumed_optimizer_steps':8,
               'main_training_steps':0,'temporary_checkpoints_only':True,'evaluator_and_TopK_CSV_JSON_readback':True,'exact_optimizer_weights_and_data_order_resume':True,
               'patience3_and_earliest_tie':True,'drive_checkpoint_upload':'not tested; checkpoint protocol used local temporary mirror'}
    finally:
        CFG,RUN,LOCAL=saved_cfg,saved_run,saved_local;sync_checkpoint=original_sync;restore_rng(saved_rng)

def pre_training_validation(source,store,frozen,base,hashes):
    global READY_FOR_TRAINING,_VALIDATION_RECEIPT
    READY_FOR_TRAINING=False;_VALIDATION_RECEIPT=None
    before_rng=rng_state();report={'passed':[],'failed':[],'skipped':[],'ready_for_training':False,'session_id':VALIDATION_SESSION_ID,
        'required_checks':['environment_encoder','coverage_cache','baseline','real_batch_smoke','production_runner_smoke','frozen_weights_rng'],'checks':{}}
    def record(name,value):report['checks'][name]='passed';report['passed'].append({name:value})
    try:
        if DEVICE.type!='cuda':
            report['checks']['environment_encoder']='skipped';report['skipped'].append('Real CUDA encoder/runtime validation required; CPU unit tests cannot mark this notebook ready')
            raise RuntimeError('GPU validation pending; select a Colab GPU and run from the top')
        require(BOOTSTRAP['passed'] and not BOOTSTRAP.get('dependency_scope',{}).get('critical_conflicts'),'Environment bootstrap not passed')
        bp=read_json(RUN/'base_preflight.json');require(bp['passed'] and bp['strict_load'] and bp['native_forward_parity'] and bp['checkpoint_sha256']==EXPECTED_BASE_MARKER['sha256'] and bp['hashes']==hashes,'Encoder strict/native preflight missing')
        expected_device=torch.device('cuda',torch.cuda.current_device()) if DEVICE.type=='cuda' else DEVICE
        require(all(p.dtype==torch.float32 and p.device==expected_device for m in (base,frozen) for p in m.parameters()),'Encoder dtype/device mismatch')
        assert_backbones_frozen(base,frozen,hashes);record('environment_encoder',{'strict_loading':True,'device':str(DEVICE),'dtype':'float32'})
        seal=seal_cache(store,source);record('coverage_cache',{'train':16204,'val':1796,'cached_queries':18000,'verified_local_shards':len(seal)})
        baseline=read_json(RUN/'baseline'/'verification.json');require(baseline['passed'] and baseline['metrics']['n_query']==1796 and baseline['checkpoint_sha256']==EXPECTED_BASE_MARKER['sha256'],'Full baseline not reproduced')
        require(max(abs(v) for v in baseline['deviation_pp'].values())<=CFG['baseline_max_metric_deviation_pp'],'Baseline deviation exceeds fixed tolerance')
        record('baseline',{'Mean':baseline['metrics']['Mean'],'tolerance_pp':CFG['baseline_max_metric_deviation_pp']})
        batch_report=preflight_development(source,store,frozen,base,hashes);require(not batch_report['failed'],'Real data preflight failed');record('real_batch_smoke',batch_report)
        require(batch_report['locked_batch']==CFG['batch_size'],'Validated batch differs from config')
        report['locked_batch']=batch_report['locked_batch']
        smoke=isolated_training_smoke(source,store,frozen);record('production_runner_smoke',smoke)
        assert_backbones_frozen(base,frozen,hashes)
        after_rng=rng_state();require(before_rng['python']==after_rng['python'] and np.array_equal(before_rng['numpy'][1],after_rng['numpy'][1]) and before_rng['numpy'][2:]==after_rng['numpy'][2:] and torch.equal(before_rng['torch'],after_rng['torch']),'Validation changed main RNG')
        if before_rng['cuda'] is not None:require(all(torch.equal(a,b) for a,b in zip(before_rng['cuda'],after_rng['cuda'])),'Validation changed main CUDA RNG')
        record('frozen_weights_rng',{'backbones_unchanged':True,'main_initialization_RNG_unchanged':True})
        require(all(report['checks'].get(name)=='passed' for name in report['required_checks']),'Some required checks have not passed')
        binding=validation_binding(source,store,hashes)
        _VALIDATION_RECEIPT={'context_sha256':fingerprint(binding),'object_ids':(id(source),id(store),id(frozen),id(base)),'local_cache_seal':seal}
        report['context_sha256']=_VALIDATION_RECEIPT['context_sha256'];report['ready_for_training']=True;READY_FOR_TRAINING=True
    except BaseException:
        report['failed'].append(traceback.format_exc());raise
    finally:
        # A failed/partial run can never leave an old ready flag or ready report behind.
        if not report['ready_for_training']:READY_FOR_TRAINING=False;_VALIDATION_RECEIPT=None
        for name in report['required_checks']:
            if name not in report['checks']:
                report['checks'][name]='skipped';report['skipped'].append(name+': not reached because validation stopped')
        restore_rng(before_rng);write_json(RUN/'test_report.json',report)
    print('ready_for_training = True; no full training was run.');return report

def require_ready_for_training(source,store,frozen,base,hashes):
    require(READY_FOR_TRAINING and _VALIDATION_RECEIPT is not None,'Run Pre-training validation successfully in this runtime first')
    report=read_json(RUN/'test_report.json')
    require(report.get('ready_for_training') is True and not report['failed'] and all(report['checks'].get(k)=='passed' for k in report['required_checks']),'Required validation is incomplete')
    require(report.get('locked_batch')==CFG['batch_size'],'Validated batch size missing or changed; rerun Pre-training validation')
    require(report.get('session_id')==VALIDATION_SESSION_ID and _VALIDATION_RECEIPT['object_ids']==(id(source),id(store),id(frozen),id(base)),'Ready report belongs to another runtime/object set')
    require(report['context_sha256']==_VALIDATION_RECEIPT['context_sha256']==fingerprint(validation_binding(source,store,hashes)),'Config/source/cache changed after validation')
    for p,sig in _VALIDATION_RECEIPT['local_cache_seal'].items():
        path=Path(p);require(path.is_file() and (path.stat().st_size,path.stat().st_mtime_ns)==sig,'Cache changed after validation; rerun cache preflight')
    assert_backbones_frozen(base,frozen,hashes)


## 2. Kiểm kê và strict checkpoint
Nạp đúng conventional best seed42, revision/tokenizer/processor/weights đã khóa; strict-load schema cũ rồi tách trained text/projection, vision/text/projection đều eval và requires_grad=False. Forward thật và native/extracted parity trước cache. Checkpoint adapter trial chỉ được strict-load **để đọc kiểm chứng nguồn**, không warm-start. Validation/gallery của trial phải khớp manifest hiện tại mới so chất lượng trực tiếp.


In [ ]:
init_development()
with timed_phase('source_checkpoint'),stage('source_checkpoint'):
    SOURCE=inspect_sources()
    BASE,FROZEN,TOKENIZER,PROCESSOR,BACKBONE_HASHES=load_frozen_checkpoint(SOURCE)
    TRIAL_REFERENCE=inspect_trial_artifacts(SOURCE,FROZEN)
    initial_probe,initial_sha,parameter_count=ensure_fresh_adapter(FROZEN.projection.weight)
    write_json(RUN/'initialization_contract.json',{'initial_sha256':initial_sha,'parameters':parameter_count,'base_seed':CFG['base_seed'],'adapter_seed':CFG['adapter_seed'],'warm_start':False},immutable=True)
    del initial_probe
    assert_backbones_frozen(BASE,FROZEN,BACKBONE_HASHES)
print('Train/val:',len(SOURCE['train']),len(SOURCE['validation']),'| verified historical trial:',TRIAL_REFERENCE['metrics']['Mean'])


## 3. Cache đầy đủ, reuse theo ID và extraction contract
Text sequence chỉ từ trained conventional; image chỉ final reference tokens/pooled và target/gallery cùng không gian. Không cache12vision layers. Cache mới khóa full manifest/ID order/SHA/precision/revision/preprocessing; cache trial chỉ đọc. Old notebook code/version/subset khác là dự kiến: chỉ converter đã khóa và **tất cả trường extraction vật lý khớp** mới reuse từng ID. Runtime/extraction khác thì không reuse và ghi lý do; shard nguồn hỏng thì dừng, không bỏ checksum. Encode phần thiếu; memmap local sau preflight, không JSON/hash Drive trong batch.

Cache text gồm cả train và validation để inference; chỉ16.204 query dùng cho optimizer. Số byte/disk được kiểm tra trước encode. Báo riêng chi phí cache và số hàng tái dùng, không gọi compute-only là pipeline runtime.


In [ ]:
with timed_phase('cache'),stage('cache'):
    started=time.perf_counter()
    CACHE=prepare_cache(SOURCE,BASE,FROZEN,TOKENIZER)
    write_json(RUN/'cache_timing.json',{'elapsed_seconds':time.perf_counter()-started,'fingerprint':CACHE.key,'scope':'metadata/verify/copy/reuse/missing encode, no adapter training'})


## 4. Baseline và batch preflight
Chạy baseline đủ1796 với tolerance0,01 điểm phần trăm như cũ. Nếu lệch thì dừng. Batch preflight dùng model/RNG tạm, kiểm gate-off, gradient theo gate zero, teacher detach, padding/mask/EOS, duplicate positives, ranking/macro và state save/load. Cell **Pre-training validation** bên dưới bổ sung production runner smoke, evaluator/TopK/CSV/JSON, early stopping và exact resume; chỉ ready khi đủ mọi check trong cùng runtime.


In [ ]:
with timed_phase('baseline'),stage('baseline'):
    BASELINE_REPORT=verify_baseline(SOURCE,CACHE,FROZEN)
    assert_backbones_frozen(BASE,FROZEN,BACKBONE_HASHES)
with timed_phase('preflight'),stage('preflight'):
    TEST_REPORT=preflight_development(SOURCE,CACHE,FROZEN,BASE,BACKBONE_HASHES)


In [ ]:
# Chẩn đoán lỗi storage_tests trên chính runtime; không sửa cache/model/RNG chính.
_diag_rng = rng_state()
try:
    from transformers.generation import GenerationMixin
    _diag = {'device':str(DEVICE), 'torch':torch.__version__, 'generation_import':True, 'base_preflight':read_json(RUN/'base_preflight.json'), 'cache_audit':read_json(RUN/'cache_audit.json'), 'baseline':read_json(RUN/'baseline'/'verification.json')}
    with tempfile.TemporaryDirectory(prefix='storage_root_cause_') as _d:
        _s=TrialStore(Path(_d)/'remote',Path(_d)/'local',{'fixture':42},{'x':{'ids':['a','b'],'shape':[3],'dtype':'float32'}},2)
        _s.prepare('x',lambda ids:np.ones((len(ids),3),np.float32))
        _chosen=next((_s.root/'x').glob('*.json'))
        _committed=_s.root/'x'/'0000000_0000002.json'
        _diag['storage']={'legacy_glob_choice':_chosen.name,'committed_marker':_committed.name,'json_files':[p.name for p in (_s.root/'x').glob('*.json')]}
        _m=read_json(_chosen);_m['fingerprint']='wrong';write_json(_chosen,_m)
        try:_s.prepare('x',lambda ids:None)
        except RuntimeError as _e:_diag['storage']['legacy_tamper_rejected']=str(_e)
        else:_diag['storage']['legacy_tamper_rejected']=False
        _diag['storage']['committed_fingerprint_unchanged']=read_json(_committed)['fingerprint']==_s.key
    _diag_dir=RUN/'diagnostics'/('storage_'+pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%SZ'))
    write_json(_diag_dir/'before_fix.json',_diag,immutable=True)
    if (RUN/'preflight_traceback.txt').exists():write_bytes(_diag_dir/'preflight_traceback.txt',(RUN/'preflight_traceback.txt').read_bytes(),immutable=True)
    print(json.dumps({'storage':_diag['storage'],'baseline_passed':_diag['baseline']['passed'],'Dev_Mean':_diag['baseline']['metrics']['Mean'],'cache_queries':_diag['cache_audit']['queries_verified'],'log':str(_diag_dir)},indent=2))
finally:
    restore_rng(_diag_rng)


## Pre-training validation — bắt buộc
Cell này không train full: dùng8 train/6val thực từ cache trong gallery chẩn đoán nhỏ, adapter/RUN/checkpoint tạm, kiểm8step + gián đoạn/resume, patience/tie và evaluator/TopK/readback. Metric tiny-gallery là fixture, không phải metric development. Khôi phục CFG/RUN/RNG sau test; không cập nhật checkpoint main. Full train bị chặn nếu thiếu check, skipped check bắt buộc, khác session/config/source/cache hoặc backbone thay đổi.


In [ ]:
with timed_phase('pre_training_validation'),stage('pre_training_validation'):
    VALIDATION_REPORT=pre_training_validation(SOURCE,CACHE,FROZEN,BASE,BACKBONE_HASHES)
print('Full training executed:',False)


In [ ]:
# Handoff: kiểm tra receipt hiện tại; không gọi optimizer hay training chính.
require_ready_for_training(SOURCE,CACHE,FROZEN,BASE,BACKBONE_HASHES)
_handoff_report=read_json(RUN/'test_report.json')
_handoff_baseline=read_json(RUN/'baseline'/'verification.json')
print(json.dumps({'ready_for_training':READY_FOR_TRAINING,'required_checks':_handoff_report['checks'],'frozen_baseline_Dev_Mean_percent':_handoff_baseline['metrics']['Mean'],'development_train':len(SOURCE['train']),'development_val':len(SOURCE['validation']),'verified_cache_shards':len(CACHE.paths),'RUN_FULL_TRAINING':RUN_FULL_TRAINING,'main_checkpoint_files':[p.name for p in RUN.glob('*.pt')],'run_id':CFG['experiment_id']},indent=2))
print('Để bắt đầu: đặt RUN_FULL_TRAINING=True trong cell riêng, rồi chạy cell training ở mục 5. Runtime mới: chạy lại từ đầu đến Pre-training validation.')


## 5. Training chỉ sau validation
Mặc định chưa train. Chạy từ đầu đến **Pre-training validation**; xem `test_report.json` có passed/failed/skipped và ready_for_training. Nếu ready=True và muốn bắt đầu, đặt `RUN_FULL_TRAINING=True` trong một cell rồi chạy riêng cell training. Đừng chạy lại cell cấu hình vì nó đưa flag về False; nếu init/runtime/source/cache đổi thì phải validation lại.

Max10epoch, patience3, batch32, FP32, constant LR giữ nguyên. Chỉ resume checkpoint cùng run/fingerprints; bản checked dùng run_id mới, không lấy trọng số trial hoặc ghi đè run cũ. Đồng bộ local atomic→Drive mỗi64step/cuối epoch, tối đa63step chưa sync có thể mất. Best chỉ từ epoch adapter đã train, baseline không là ứng viên. Summary chỉ finished khi best/last và toàn bộ metric/artifact được hash/coverage xác minh.


In [ ]:
RUN_FULL_TRAINING=True

In [ ]:
if RUN_FULL_TRAINING:
    require_ready_for_training(SOURCE,CACHE,FROZEN,BASE,BACKBONE_HASHES)
    with timed_phase('train_development'),stage('train_development'):
        DEVELOPMENT_STATE=train_development(SOURCE,CACHE,FROZEN,BASE,BACKBONE_HASHES)
else:
    print('Validation-only: full training chưa chạy. Sau khi Pre-training validation đạt, đặt RUN_FULL_TRAINING=True và chạy riêng cell training.')


## 6. Kết quả và export opt-in
Khi có best trained checkpoint, rerank/readback full-val, CSV per-category/per-query/Top50. Chưa có kết quả thì để null; status.finished một mình không đủ để báo complete. Summary giữ delta âm và khác biệt ngân sách trial2048 so với development16204. API enhanced sequence/mask/EOS giữ nguyên, chưa gọi export toàn bộ hoặc train head.

Runtime mới: chọn GPU, chạy từ đầu đến Pre-training validation. Không bỏ check khi fail, không đổi tolerance/precision để vượt lỗi. Chi tiết trong test_report.json; local report có GPU/Drive mới skipped nên ready_for_training=False cho đến khi cell validation thực tế đạt.


In [ ]:
if (RUN/'best.pt').is_file():
    with timed_phase('best_export_readback'),stage('best_export_readback'):
        BEST_REPORT=export_best_results(SOURCE,CACHE,FROZEN)
        assert_backbones_frozen(BASE,FROZEN,BACKBONE_HASHES)
SUMMARY=development_summary()
print('Verified experiment complete:',SUMMARY['finished'],'| ready in this runtime:',READY_FOR_TRAINING)
